# PHASE 3.4 — RESNET18 BASELINE MODEL

In [2]:
# ============================================================
# CISLR v1.5-a
# PHASE 3.4 — RESNET18 FRAME-LEVEL BASELINE
# COMPLETE ROBUST VERSION WITH CORRUPTED-VIDEO HANDLING
# ============================================================
#
# PURPOSE
# -------
# This cell establishes the spatial/frame-level ResNet18
# baseline for CISLR v1.5-a.
#
# PIPELINE
#
#     Verified training metadata
#                ↓
#     Video integrity validation
#                ↓
#       Corrupted video?
#          /          \
#        YES           NO
#         ↓             ↓
#     Exclude UID    Keep video
#                       ↓
#                Frame dataset
#                       ↓
#                    ResNet18
#                       ↓
#                    512-D
#                       ↓
#                 Linear layer
#                       ↓
#                  4764 classes
#                       ↓
#                    Softmax
#                       ↓
#                 Predicted gloss
#
#
# WHY WE DROP THE ENTIRE VIDEO
# ----------------------------
# If one frame in a video cannot be decoded, keeping the other
# frames would produce an inconsistent subset of that video.
#
# More importantly, the same video may later be used by the
# temporal model, where missing one of the required frames makes
# the complete sequence invalid.
#
# Therefore the conservative research choice is:
#
#     ANY REQUIRED FRAME FAILURE
#                  ↓
#          EXCLUDE ENTIRE UID
#
# This does NOT modify or delete the original MP4.
#
#
# IMPORTANT
# ---------
# The official CISLR test set is NOT used here.
#
# Only the verified training frame metadata is used.
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd

import cv2

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 3. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("DEVICE")
print("=" * 80)

print("Device:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0)
            .total_memory / (1024 ** 3),
            2
        ),
        "GB"
    )


# ============================================================
# 4. CANONICAL PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
).resolve()

PROCESSED_DIR = (
    PROJECT_ROOT / "processed"
)

MODEL_DIR = (
    PROJECT_ROOT /
    "models" /
    "baseline"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not PROJECT_ROOT.is_dir():

    raise FileNotFoundError(
        f"Project root does not exist:\n"
        f"{PROJECT_ROOT}"
    )

if not PROCESSED_DIR.is_dir():

    raise FileNotFoundError(
        f"Processed directory does not exist:\n"
        f"{PROCESSED_DIR}"
    )


print()
print("=" * 80)
print("PROJECT PATHS")
print("=" * 80)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Processed:",
    PROCESSED_DIR
)

print(
    "Model directory:",
    MODEL_DIR
)


# ============================================================
# 5. CONFIGURATION
# ============================================================

IMAGE_SIZE = 224

BATCH_SIZE = 32

NUM_EPOCHS = 10

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

PIN_MEMORY = torch.cuda.is_available()

IMAGE_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGE_STD = [
    0.229,
    0.224,
    0.225
]


# ============================================================
# 6. LOAD LABEL MAPPING
# ============================================================

LABEL_MAPPING_PATH = (
    PROCESSED_DIR /
    "stage3_label_mapping.csv"
)

if not LABEL_MAPPING_PATH.exists():

    raise FileNotFoundError(
        f"Label mapping not found:\n"
        f"{LABEL_MAPPING_PATH}"
    )

label_mapping = pd.read_csv(
    LABEL_MAPPING_PATH
)

required_mapping_columns = {
    "label",
    "gloss"
}

missing_mapping_columns = (
    required_mapping_columns -
    set(label_mapping.columns)
)

if missing_mapping_columns:

    raise ValueError(
        "Missing label mapping columns:\n"
        f"{missing_mapping_columns}"
    )

label_mapping = (
    label_mapping
    .sort_values("label")
    .reset_index(drop=True)
)

NUM_CLASSES = len(
    label_mapping
)

index_to_label = dict(
    zip(
        label_mapping["label"].astype(int),
        label_mapping["gloss"].astype(str)
    )
)

label_to_index = {
    gloss: label
    for label, gloss
    in index_to_label.items()
}

print()
print("=" * 80)
print("LABEL MAPPING")
print("=" * 80)

print(
    "Number of classes:",
    NUM_CLASSES
)

print(
    "Expected CISLR classes:",
    4764
)

if NUM_CLASSES != 4764:

    print(
        "WARNING: Expected 4764 classes, "
        f"but found {NUM_CLASSES}."
    )


# ============================================================
# 7. LOAD TRAINING FRAME METADATA
# ============================================================

TRAIN_METADATA_PATH = (
    PROCESSED_DIR /
    "stage3_train_frame_metadata.csv"
)

if not TRAIN_METADATA_PATH.exists():

    raise FileNotFoundError(
        f"Training metadata not found:\n"
        f"{TRAIN_METADATA_PATH}"
    )

frame_df = pd.read_csv(
    TRAIN_METADATA_PATH
)

print()
print("=" * 80)
print("RAW TRAINING FRAME METADATA")
print("=" * 80)

print(
    "Rows:",
    len(frame_df)
)

print(
    "Columns:",
    list(frame_df.columns)
)


# ============================================================
# 8. VALIDATE METADATA
# ============================================================

required_columns = {
    "uid",
    "gloss",
    "video_path",
    "sample_number",
    "frame_index"
}

missing_columns = (
    required_columns -
    set(frame_df.columns)
)

if missing_columns:

    raise ValueError(
        "Missing required metadata columns:\n"
        f"{missing_columns}"
    )


# ============================================================
# 9. REMOVE INVALID LABEL ROWS
# ============================================================
#
# Raw metadata is NOT modified.
# We only create an in-memory cleaned dataframe.
# ============================================================

initial_rows = len(
    frame_df
)

frame_df = frame_df[
    frame_df["gloss"].notna()
].copy()

frame_df["gloss"] = (
    frame_df["gloss"]
    .astype(str)
)

frame_df["video_path"] = (
    frame_df["video_path"]
    .astype(str)
)

frame_df["frame_index"] = (
    frame_df["frame_index"]
    .astype(int)
)

frame_df["label"] = (
    frame_df["gloss"]
    .map(label_to_index)
)

if frame_df["label"].isna().any():

    unknown_glosses = (
        frame_df.loc[
            frame_df["label"].isna(),
            "gloss"
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "Unknown glosses found:\n"
        f"{unknown_glosses[:20]}"
    )

frame_df["label"] = (
    frame_df["label"]
    .astype(int)
)

print()
print(
    "Rows removed due to invalid gloss:",
    initial_rows - len(frame_df)
)


# ============================================================
# 10. VIDEO INTEGRITY SCAN
# ============================================================
#
# THIS IS THE IMPORTANT FIX.
#
# We validate every video BEFORE creating the DataLoader.
#
# For every UID:
#
#     1. Find its video.
#     2. Check that the video opens.
#     3. Try decoding every frame referenced by the metadata.
#     4. If ANY referenced frame fails:
#            → mark entire UID as invalid.
#
# This prevents corrupted MP4s from crashing training.
# ============================================================

print()
print("=" * 80)
print("VIDEO INTEGRITY SCAN")
print("=" * 80)

video_integrity_records = []

valid_uids = []

invalid_uids = []

unique_uids = (
    frame_df["uid"]
    .drop_duplicates()
    .tolist()
)

print(
    "Videos to validate:",
    len(unique_uids)
)

scan_start = time.time()


for counter, uid in enumerate(
    unique_uids,
    start=1
):

    uid_rows = frame_df[
        frame_df["uid"] == uid
    ]

    video_path = (
        uid_rows["video_path"]
        .iloc[0]
    )

    failure_reason = ""

    failed_frame = None

    # --------------------------------------------------------
    # Check file existence
    # --------------------------------------------------------

    if not Path(video_path).exists():

        failure_reason = (
            "video_file_missing"
        )

    else:

        # ----------------------------------------------------
        # Open video
        # ----------------------------------------------------

        cap = cv2.VideoCapture(
            video_path
        )

        if not cap.isOpened():

            failure_reason = (
                "video_open_failed"
            )

        else:

            # ------------------------------------------------
            # Decode all referenced frames
            # ------------------------------------------------

            try:

                for frame_index in (
                    uid_rows["frame_index"]
                    .tolist()
                ):

                    cap.set(
                        cv2.CAP_PROP_POS_FRAMES,
                        int(frame_index)
                    )

                    success, frame = (
                        cap.read()
                    )

                    if (
                        not success
                        or frame is None
                        or frame.size == 0
                    ):

                        failed_frame = int(
                            frame_index
                        )

                        failure_reason = (
                            "frame_decode_failed"
                        )

                        break

            finally:

                cap.release()


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    if failure_reason:

        invalid_uids.append(
            uid
        )

        video_integrity_records.append({

            "uid": uid,

            "video_path": video_path,

            "status": "INVALID",

            "failure_reason":
                failure_reason,

            "failed_frame":
                failed_frame,

            "num_metadata_frames":
                len(uid_rows)

        })

    else:

        valid_uids.append(
            uid
        )

        video_integrity_records.append({

            "uid": uid,

            "video_path": video_path,

            "status": "VALID",

            "failure_reason": "",

            "failed_frame": "",

            "num_metadata_frames":
                len(uid_rows)

        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        counter % 100 == 0
        or counter == len(unique_uids)
    ):

        print(
            f"Checked "
            f"{counter:,}/{len(unique_uids):,} "
            f"videos | "
            f"Invalid: {len(invalid_uids):,}"
        )


scan_elapsed = (
    time.time()
    - scan_start
)

integrity_df = pd.DataFrame(
    video_integrity_records
)


# ============================================================
# 11. SAVE CORRUPTION REPORT
# ============================================================

CORRUPTION_REPORT_PATH = (
    PROCESSED_DIR /
    "stage3_4_resnet18_video_integrity.csv"
)

integrity_df.to_csv(
    CORRUPTION_REPORT_PATH,
    index=False
)

print()
print("=" * 80)
print("VIDEO INTEGRITY RESULT")
print("=" * 80)

print(
    "Total videos checked:",
    len(unique_uids)
)

print(
    "Valid videos:",
    len(valid_uids)
)

print(
    "Invalid videos:",
    len(invalid_uids)
)

print(
    "Scan time:",
    f"{scan_elapsed / 60:.2f} minutes"
)

print(
    "Report:",
    CORRUPTION_REPORT_PATH
)


# ============================================================
# 12. DISPLAY INVALID VIDEOS
# ============================================================

if len(invalid_uids) > 0:

    print()
    print("=" * 80)
    print("INVALID VIDEOS")
    print("=" * 80)

    invalid_report = integrity_df[
        integrity_df["status"] == "INVALID"
    ]

    print(
        invalid_report[
            [
                "uid",
                "video_path",
                "failure_reason",
                "failed_frame"
            ]
        ]
        .to_string(index=False)
    )

else:

    print()
    print(
        "✓ No corrupted training videos detected."
    )


# ============================================================
# 13. REMOVE ALL ROWS BELONGING TO INVALID UIDS
# ============================================================
#
# IMPORTANT:
#
# We remove the COMPLETE UID, not just the bad frame.
#
# Example:
#
#     yffUIRSERd0_2.mp4
#             ↓
#       frame 128 fails
#             ↓
#       entire UID excluded
#
# Raw MP4 remains untouched.
# ============================================================

before_filter_rows = len(
    frame_df
)

before_filter_uids = (
    frame_df["uid"]
    .nunique()
)

frame_df = frame_df[
    ~frame_df["uid"].isin(
        invalid_uids
    )
].copy()

after_filter_rows = len(
    frame_df
)

after_filter_uids = (
    frame_df["uid"]
    .nunique()
)

print()
print("=" * 80)
print("CORRUPTED DATA REMOVAL")
print("=" * 80)

print(
    "Rows before:",
    before_filter_rows
)

print(
    "Rows after:",
    after_filter_rows
)

print(
    "Rows removed:",
    before_filter_rows -
    after_filter_rows
)

print(
    "Videos before:",
    before_filter_uids
)

print(
    "Videos after:",
    after_filter_uids
)

print(
    "Videos removed:",
    before_filter_uids -
    after_filter_uids
)


# ============================================================
# 14. SAVE CLEANED BASELINE METADATA
# ============================================================

CLEAN_METADATA_PATH = (
    PROCESSED_DIR /
    "stage3_4_resnet18_clean_train_frames.csv"
)

frame_df.to_csv(
    CLEAN_METADATA_PATH,
    index=False
)

print()
print(
    "Clean baseline metadata saved:"
)

print(
    CLEAN_METADATA_PATH
)


# ============================================================
# 15. DATASET TRANSFORM
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGE_MEAN,
        std=IMAGE_STD
    )
])


# ============================================================
# 16. FRAME DATASET
# ============================================================

class CISLRFrameDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = transform

    def __len__(
        self
    ):

        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index
    ):

        row = (
            self.dataframe
            .iloc[index]
        )

        video_path = (
            row["video_path"]
        )

        frame_index = int(
            row["frame_index"]
        )

        label = int(
            row["label"]
        )

        cap = cv2.VideoCapture(
            video_path
        )

        if not cap.isOpened():

            raise RuntimeError(
                f"Video unexpectedly failed "
                f"after integrity validation:\n"
                f"{video_path}"
            )

        try:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                frame_index
            )

            success, frame = (
                cap.read()
            )

        finally:

            cap.release()

        if (
            not success
            or frame is None
            or frame.size == 0
        ):

            raise RuntimeError(
                f"Frame unexpectedly failed "
                f"after integrity validation:\n"
                f"Video: {video_path}\n"
                f"Frame: {frame_index}"
            )

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        image = Image.fromarray(
            frame
        )

        if self.transform is not None:

            image = self.transform(
                image
            )

        return (
            image,
            torch.tensor(
                label,
                dtype=torch.long
            )
        )


# ============================================================
# 17. CREATE DATASET
# ============================================================

train_dataset = CISLRFrameDataset(
    frame_df,
    transform=train_transform
)

print()
print("=" * 80)
print("CLEAN RESNET18 DATASET")
print("=" * 80)

print(
    "Frame samples:",
    len(train_dataset)
)

print(
    "Videos:",
    frame_df["uid"].nunique()
)

print(
    "Classes represented:",
    frame_df["label"].nunique()
)


# ============================================================
# 18. CREATE DATALOADER
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY
)

print(
    "Batches:",
    len(train_loader)
)


# ============================================================
# 19. BATCH SANITY CHECK
# ============================================================

sample_images, sample_labels = next(
    iter(train_loader)
)

print()
print("=" * 80)
print("BATCH SANITY CHECK")
print("=" * 80)

print(
    "Images:",
    tuple(sample_images.shape)
)

print(
    "Labels:",
    tuple(sample_labels.shape)
)

if (
    sample_images.ndim != 4
    or sample_images.shape[1:]
       != (
           3,
           IMAGE_SIZE,
           IMAGE_SIZE
       )
):

    raise ValueError(
        "Incorrect ResNet18 input shape."
    )

print(
    "✓ ResNet18 input shape verified."
)


# ============================================================
# 20. RESNET18 MODEL
# ============================================================

class CISLRResNet18(
    nn.Module
):

    def __init__(
        self,
        num_classes
    ):

        super().__init__()

        self.backbone = (
            models.resnet18(
                weights=None
            )
        )

        self.backbone.fc = nn.Linear(
            512,
            num_classes
        )

    def forward(
        self,
        x
    ):

        return self.backbone(
            x
        )


# ============================================================
# 21. CREATE MODEL
# ============================================================

model = CISLRResNet18(
    num_classes=NUM_CLASSES
).to(DEVICE)

print()
print("=" * 80)
print("RESNET18")
print("=" * 80)

print(
    "Input:",
    "3 × 224 × 224"
)

print(
    "Feature vector:",
    "512-D"
)

print(
    "Output classes:",
    NUM_CLASSES
)


# ============================================================
# 22. FORWARD-PASS SANITY CHECK
# ============================================================

model.eval()

with torch.no_grad():

    sample_logits = model(
        sample_images.to(DEVICE)
    )

print()
print("=" * 80)
print("FORWARD PASS")
print("=" * 80)

print(
    "Input:",
    tuple(sample_images.shape)
)

print(
    "Output:",
    tuple(sample_logits.shape)
)

if tuple(sample_logits.shape) != (
    sample_images.shape[0],
    NUM_CLASSES
):

    raise ValueError(
        "Incorrect ResNet18 output shape."
    )

print(
    "✓ Forward pass successful."
)


# ============================================================
# 23. LOSS + OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 24. TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0

    correct = 0

    total = 0

    start_time = time.time()

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = (
            torch.argmax(
                logits,
                dim=1
            )
        )

        correct += int(
            (
                predictions == labels
            )
            .sum()
            .item()
        )

        total += (
            labels.size(0)
        )

    epoch_loss = (
        running_loss / total
    )

    epoch_accuracy = (
        correct / total
    )

    elapsed = (
        time.time()
        - start_time
    )

    return (
        epoch_loss,
        epoch_accuracy,
        elapsed
    )


# ============================================================
# 25. TRAIN RESNET18
# ============================================================

history = []

best_accuracy = 0.0

best_model_path = (
    MODEL_DIR /
    "resnet18_baseline_best.pt"
)

print()
print("=" * 80)
print("STARTING RESNET18 BASELINE TRAINING")
print("=" * 80)

for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    train_loss, train_accuracy, elapsed = (
        train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            DEVICE
        )
    )

    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_accuracy,

        "time_seconds":
            elapsed

    })

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_accuracy:.4f} | "
        f"Time: {elapsed / 60:.2f} min"
    )

    if train_accuracy > best_accuracy:

        best_accuracy = (
            train_accuracy
        )

        torch.save({

            "model_state_dict":
                model.state_dict(),

            "num_classes":
                NUM_CLASSES,

            "architecture":
                "ResNet18",

            "best_train_accuracy":
                best_accuracy,

            "label_mapping":
                index_to_label,

            "image_size":
                IMAGE_SIZE,

            "normalization_mean":
                IMAGE_MEAN,

            "normalization_std":
                IMAGE_STD

        }, best_model_path)

        print(
            "   ✓ Best model saved."
        )


# ============================================================
# 26. SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)

history_path = (
    MODEL_DIR /
    "resnet18_baseline_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)


# ============================================================
# 27. FINAL RESULT
# ============================================================

print()
print("=" * 80)
print("PHASE 3.4 — RESNET18 BASELINE COMPLETE")
print("=" * 80)

print(
    "Architecture:",
    "ResNet18"
)

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "Clean training videos:",
    after_filter_uids
)

print(
    "Excluded corrupted videos:",
    len(invalid_uids)
)

print(
    "Clean training frames:",
    after_filter_rows
)

print(
    "Best training accuracy:",
    f"{best_accuracy:.4f}"
)

print()
print(
    "Best model:"
)

print(
    best_model_path
)

print()
print(
    "Training history:"
)

print(
    history_path
)

print()
print(
    "Integrity report:"
)

print(
    CORRUPTION_REPORT_PATH
)

print()
print(
    "Clean metadata:"
)

print(
    CLEAN_METADATA_PATH
)

print()
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

print(
    "ResNet18 provides the spatial/frame-level baseline."
)

print(
    "It uses appearance information from individual frames "
    "but does not model temporal motion or frame ordering."
)

print(
    "Corrupted videos were excluded before training."
)

print(
    "Original raw MP4 files were not modified."
)

print(
    "The official CISLR test set remains untouched."
)

print("=" * 80)

DEVICE
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU Memory: 5.64 GB

PROJECT PATHS
Project root: /home/dipshikha/Music/cao/CISLR_RESEARCH
Processed: /home/dipshikha/Music/cao/CISLR_RESEARCH/processed
Model directory: /home/dipshikha/Music/cao/CISLR_RESEARCH/models/baseline

LABEL MAPPING
Number of classes: 4764
Expected CISLR classes: 4764

RAW TRAINING FRAME METADATA
Rows: 38112
Columns: ['uid', 'gloss', 'video_path', 'split', 'sample_number', 'frame_index', 'fps', 'frame_count', 'width', 'height', 'duration_seconds']

Rows removed due to invalid gloss: 0

VIDEO INTEGRITY SCAN
Videos to validate: 4764
Checked 100/4,764 videos | Invalid: 4
Checked 200/4,764 videos | Invalid: 12
Checked 300/4,764 videos | Invalid: 14
Checked 400/4,764 videos | Invalid: 20
Checked 500/4,764 videos | Invalid: 26
Checked 600/4,764 videos | Invalid: 30
Checked 700/4,764 videos | Invalid: 40
Checked 800/4,764 videos | Invalid: 48
Checked 900/4,764 videos | Invalid: 58
Checked 1,000/4,764 videos |

# PHASE 3.4 — selecting efficient one


In [9]:
# =============================================================================
# CISLR v1.5-a
# CORRECTED POST-HOC SPATIAL BACKBONE EVALUATION
# =============================================================================
#
# PURPOSE
# -------
# The previous evaluation produced ~0% accuracy because the checkpoint
# parameter names contained "backbone." while the freshly-created evaluation
# models expected the CNN parameters without that prefix.
#
# Example:
#
# Saved checkpoint:
#     backbone.conv1.weight
#
# Evaluation model:
#     conv1.weight
#
# Therefore the trained weights were NOT actually loaded.
#
# THIS CELL:
#   1. Does NOT train either model.
#   2. Loads the existing ResNet18 checkpoint correctly.
#   3. Loads the existing EfficientNet-B0 checkpoint correctly.
#   4. Evaluates both on the SAME 442-video subset.
#   5. Computes Top-1, Top-5 and Macro-F1.
#
# IMPORTANT SCIENTIFIC NOTE
# -------------------------
# The existing checkpoints were trained on all 4,417 clean training videos.
# Therefore the 442-video subset below was already seen during training.
# This is NOT a true held-out validation experiment.
#
# We therefore call this:
#     POST-HOC SPATIAL BACKBONE EVALUATION
#
# It is useful for checking the existing models, but should not be reported
# as an independent validation result in a paper.
# =============================================================================


# -----------------------------------------------------------------------------
# 1. IMPORTS
# -----------------------------------------------------------------------------

import os
import gc
import json
import time
import random
import warnings

import numpy as np
import pandas as pd

import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


# -----------------------------------------------------------------------------
# 2. REPRODUCIBILITY
# -----------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# -----------------------------------------------------------------------------
# 3. DEVICE
# -----------------------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("CISLR — CORRECTED POST-HOC SPATIAL BACKBONE EVALUATION")
print("=" * 80)

print(f"Device : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM   : "
        f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
    )

    torch.cuda.empty_cache()
    gc.collect()

    print(
        f"GPU allocated before evaluation: "
        f"{torch.cuda.memory_allocated() / (1024**3):.2f} GB"
    )

    print(
        f"GPU reserved before evaluation : "
        f"{torch.cuda.memory_reserved() / (1024**3):.2f} GB"
    )


# -----------------------------------------------------------------------------
# 4. PATHS
# -----------------------------------------------------------------------------

RESNET_CHECKPOINT = (
    "/home/dipshikha/Music/cao/"
    "CISLR_RESEARCH/models/baseline/"
    "resnet18_baseline_best.pt"
)

EFFICIENTNET_CHECKPOINT = (
    "/home/dipshikha/Music/cao/"
    "CISLR_RESEARCH/models/baseline/"
    "efficientnet_b0_baseline_best.pt"
)

CLEAN_FRAME_METADATA = (
    "/home/dipshikha/Music/cao/"
    "CISLR_RESEARCH/processed/"
    "stage3_4_resnet18_clean_train_frames.csv"
)

LABEL_MAPPING = (
    "/home/dipshikha/Music/cao/"
    "CISLR_RESEARCH/processed/"
    "stage3_label_mapping.csv"
)

OUTPUT_DIR = (
    "/home/dipshikha/Music/cao/"
    "CISLR_RESEARCH/models/baseline/"
    "scientific_spatial_validation"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# -----------------------------------------------------------------------------
# 5. CHECK FILES
# -----------------------------------------------------------------------------

required_files = {
    "ResNet18 checkpoint": RESNET_CHECKPOINT,
    "EfficientNet-B0 checkpoint": EFFICIENTNET_CHECKPOINT,
    "Clean frame metadata": CLEAN_FRAME_METADATA,
    "Label mapping": LABEL_MAPPING,
}

for name, path in required_files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

print("\nRequired files verified.")


# -----------------------------------------------------------------------------
# 6. LOAD LABEL MAPPING
# -----------------------------------------------------------------------------

label_df = pd.read_csv(LABEL_MAPPING)

print("\n" + "=" * 80)
print("LABEL MAPPING")
print("=" * 80)

print("Columns:", list(label_df.columns))
print("Shape:", label_df.shape)

if "label" not in label_df.columns or "gloss" not in label_df.columns:
    raise ValueError(
        "Label mapping must contain columns: label, gloss"
    )

label_df["label"] = pd.to_numeric(
    label_df["label"],
    errors="coerce"
)

if label_df["label"].isna().any():
    raise ValueError(
        "Label mapping contains missing/non-numeric labels."
    )

label_df["label"] = label_df["label"].astype(int)

min_label = int(label_df["label"].min())
max_label = int(label_df["label"].max())

NUM_CLASSES = max_label + 1

print("Minimum label :", min_label)
print("Maximum label :", max_label)
print("NUM_CLASSES   :", NUM_CLASSES)

expected_labels = set(range(NUM_CLASSES))
actual_labels = set(label_df["label"].tolist())

missing_labels = sorted(
    expected_labels - actual_labels
)

if missing_labels:
    raise ValueError(
        f"Missing label indices detected: {missing_labels[:20]}"
    )

index_to_label = dict(
    zip(
        label_df["label"],
        label_df["gloss"].astype(str)
    )
)

gloss_to_index = {
    gloss: int(label)
    for label, gloss in index_to_label.items()
}

print(
    "Valid gloss mappings:",
    len(gloss_to_index)
)


# -----------------------------------------------------------------------------
# 7. LOAD CLEAN FRAME METADATA
# -----------------------------------------------------------------------------

frame_df = pd.read_csv(CLEAN_FRAME_METADATA)

print("\n" + "=" * 80)
print("FRAME METADATA")
print("=" * 80)

print("Rows :", len(frame_df))
print("UIDs :", frame_df["uid"].nunique())

required_columns = [
    "uid",
    "gloss",
    "video_path",
    "frame_index",
]

for column in required_columns:
    if column not in frame_df.columns:
        raise ValueError(
            f"Missing required column: {column}"
        )


# -----------------------------------------------------------------------------
# 8. MAP GLOSS TO CLASS INDEX
# -----------------------------------------------------------------------------

frame_df["label_index"] = frame_df["gloss"].map(
    gloss_to_index
)

unmapped_rows = int(
    frame_df["label_index"].isna().sum()
)

print(
    "\nUnmapped frame rows:",
    unmapped_rows
)

if unmapped_rows > 0:
    raise ValueError(
        "Some frame rows could not be mapped to classifier labels."
    )

frame_df["label_index"] = (
    frame_df["label_index"]
    .astype(int)
)


# -----------------------------------------------------------------------------
# 9. BUILD 8-FRAME VIDEO SEQUENCES
# -----------------------------------------------------------------------------

NUM_FRAMES = 8

def get_uniform_frame_indices(frame_count, num_frames):
    """
    Select uniformly distributed frame indices.
    """
    frame_count = int(frame_count)
    num_frames = int(num_frames)

    if frame_count <= 0:
        return np.array([], dtype=np.int64)

    if num_frames <= 0:
        raise ValueError(
            "num_frames must be greater than zero."
        )

    if frame_count <= num_frames:
        return np.arange(
            frame_count,
            dtype=np.int64
        )

    indices = np.linspace(
        0,
        frame_count - 1,
        num_frames
    )

    return np.round(indices).astype(np.int64)


sequences = []

for uid, group in frame_df.groupby(
    "uid",
    sort=False
):

    group = group.sort_values(
        "frame_index"
    ).reset_index(drop=True)

    if len(group) == 0:
        continue

    video_path = str(
        group.loc[0, "video_path"]
    )

    label_index = int(
        group.loc[0, "label_index"]
    )

    gloss = str(
        group.loc[0, "gloss"]
    )

    available_indices = (
        group["frame_index"]
        .astype(int)
        .to_numpy()
    )

    selected_positions = get_uniform_frame_indices(
        len(available_indices),
        NUM_FRAMES
    )

    selected_frame_indices = (
        available_indices[selected_positions]
    )

    if len(selected_frame_indices) != NUM_FRAMES:
        continue

    sequences.append(
        {
            "uid": uid,
            "gloss": gloss,
            "video_path": video_path,
            "frame_indices": selected_frame_indices.tolist(),
            "label_index": label_index,
        }
    )


print("\n" + "=" * 80)
print("VIDEO SEQUENCES")
print("=" * 80)

print(
    "Usable sequences:",
    len(sequences)
)


# -----------------------------------------------------------------------------
# 10. FILE EXISTENCE CHECK
# -----------------------------------------------------------------------------

valid_sequences = []

for sequence in sequences:

    if not os.path.exists(
        sequence["video_path"]
    ):
        continue

    valid_sequences.append(
        sequence
    )

sequences = valid_sequences

print(
    "Sequences after file check:",
    len(sequences)
)


# -----------------------------------------------------------------------------
# 11. RECREATE THE SAME 90/10 VIDEO-LEVEL SPLIT
# -----------------------------------------------------------------------------

all_indices = np.arange(
    len(sequences)
)

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=SEED,
    shuffle=True
)

posthoc_validation_sequences = [
    sequences[i]
    for i in validation_indices
]

posthoc_training_sequences = [
    sequences[i]
    for i in train_indices
]

training_uids = {
    x["uid"]
    for x in posthoc_training_sequences
}

validation_uids = {
    x["uid"]
    for x in posthoc_validation_sequences
}

uid_overlap = (
    training_uids.intersection(
        validation_uids
    )
)

print("\n" + "=" * 80)
print("POST-HOC VIDEO SUBSET")
print("=" * 80)

print(
    "Reference training subset:",
    len(posthoc_training_sequences)
)

print(
    "Evaluation videos:",
    len(posthoc_validation_sequences)
)

print(
    "UID overlap:",
    len(uid_overlap)
)

if len(uid_overlap) > 0:
    raise RuntimeError(
        "UID overlap detected."
    )


# -----------------------------------------------------------------------------
# 12. SAVE THE FIXED SPLIT
# -----------------------------------------------------------------------------

split_records = []

for sequence in posthoc_training_sequences:
    split_records.append(
        {
            "uid": sequence["uid"],
            "split": "reference_train_90pct"
        }
    )

for sequence in posthoc_validation_sequences:
    split_records.append(
        {
            "uid": sequence["uid"],
            "split": "posthoc_evaluation_10pct"
        }
    )

split_df = pd.DataFrame(
    split_records
)

split_path = os.path.join(
    OUTPUT_DIR,
    "scientific_video_split.csv"
)

split_df.to_csv(
    split_path,
    index=False
)

print(
    "\nSaved split:",
    split_path
)


# -----------------------------------------------------------------------------
# 13. IMAGE PREPROCESSING
# -----------------------------------------------------------------------------
#
# Must match the original baseline preprocessing:
#
#   RGB
#   ↓
#   Resize 224 × 224
#   ↓
#   ToTensor
#   ↓
#   ImageNet normalization
# -----------------------------------------------------------------------------

IMAGE_SIZE = 224

eval_transform = transforms.Compose(
    [
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE)
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406
            ],
            std=[
                0.229,
                0.224,
                0.225
            ]
        ),
    ]
)


# -----------------------------------------------------------------------------
# 14. VIDEO DATASET
# -----------------------------------------------------------------------------

class SpatialVideoDataset(Dataset):

    def __init__(
        self,
        sequences,
        transform=None
    ):
        self.sequences = sequences
        self.transform = transform

    def __len__(self):
        return len(self.sequences)

    def _read_frame(
        self,
        video_path,
        frame_index
    ):

        cap = cv2.VideoCapture(
            video_path
        )

        if not cap.isOpened():
            raise RuntimeError(
                f"Could not open video:\n"
                f"{video_path}"
            )

        try:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_index)
            )

            success, frame = cap.read()

        finally:

            cap.release()

        if (
            not success
            or frame is None
            or frame.size == 0
        ):
            raise RuntimeError(
                f"Could not decode frame "
                f"{frame_index} from:\n"
                f"{video_path}"
            )

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        image = Image.fromarray(
            frame
        )

        return image

    def __getitem__(
        self,
        index
    ):

        sequence = self.sequences[index]

        frames = []

        for frame_index in sequence[
            "frame_indices"
        ]:

            image = self._read_frame(
                sequence["video_path"],
                frame_index
            )

            if self.transform is not None:
                image = self.transform(
                    image
                )

            frames.append(image)

        frames = torch.stack(
            frames,
            dim=0
        )

        label = torch.tensor(
            sequence["label_index"],
            dtype=torch.long
        )

        return (
            frames,
            label
        )


# -----------------------------------------------------------------------------
# 15. VALIDATION LOADER
# -----------------------------------------------------------------------------

VIDEO_BATCH_SIZE = 2

evaluation_dataset = SpatialVideoDataset(
    posthoc_validation_sequences,
    transform=eval_transform
)

evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=VIDEO_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("\n" + "=" * 80)
print("EVALUATION LOADER")
print("=" * 80)

print(
    "Evaluation videos:",
    len(evaluation_dataset)
)

print(
    "Video batch size:",
    VIDEO_BATCH_SIZE
)

print(
    "Frames/batch:",
    VIDEO_BATCH_SIZE * NUM_FRAMES
)


# -----------------------------------------------------------------------------
# 16. CHECKPOINT LOADING FUNCTION
# -----------------------------------------------------------------------------
#
# IMPORTANT:
#
# Existing checkpoints use:
#
#     backbone.conv1.weight
#     backbone.layer1...
#
# while torchvision models use:
#
#     conv1.weight
#     layer1...
#
# Therefore we explicitly remove the "backbone." prefix.
#
# We also handle possible checkpoint structures:
#
#   checkpoint["model_state_dict"]
#   checkpoint["state_dict"]
#   checkpoint itself being a state_dict
# -----------------------------------------------------------------------------

def extract_state_dict(checkpoint):

    if isinstance(
        checkpoint,
        dict
    ):

        if "model_state_dict" in checkpoint:
            return checkpoint[
                "model_state_dict"
            ]

        if "state_dict" in checkpoint:
            return checkpoint[
                "state_dict"
            ]

        # If it already looks like a state dict
        if all(
            isinstance(k, str)
            for k in checkpoint.keys()
        ):
            return checkpoint

    raise ValueError(
        "Could not identify model state_dict "
        "inside checkpoint."
    )


def prepare_state_dict_for_torchvision(
    state_dict
):

    cleaned_state_dict = {}

    for key, value in state_dict.items():

        new_key = key

        # Remove DataParallel prefix if present
        if new_key.startswith(
            "module."
        ):
            new_key = new_key[
                len("module.") :
            ]

        # Remove custom model backbone prefix
        if new_key.startswith(
            "backbone."
        ):
            new_key = new_key[
                len("backbone.") :
            ]

        cleaned_state_dict[
            new_key
        ] = value

    return cleaned_state_dict


def load_checkpoint_correctly(
    model,
    checkpoint_path,
    model_name
):

    print(
        f"\nLoading {model_name} checkpoint..."
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    state_dict = extract_state_dict(
        checkpoint
    )

    cleaned_state_dict = (
        prepare_state_dict_for_torchvision(
            state_dict
        )
    )

    # ---------------------------------------------------------
    # Validate key compatibility BEFORE loading.
    # ---------------------------------------------------------

    model_keys = set(
        model.state_dict().keys()
    )

    checkpoint_keys = set(
        cleaned_state_dict.keys()
    )

    missing = sorted(
        model_keys - checkpoint_keys
    )

    unexpected = sorted(
        checkpoint_keys - model_keys
    )

    matched = (
        len(
            model_keys.intersection(
                checkpoint_keys
            )
        )
    )

    print(
        "Checkpoint keys:",
        len(checkpoint_keys)
    )

    print(
        "Model keys:",
        len(model_keys)
    )

    print(
        "Matched keys:",
        matched
    )

    print(
        "Missing keys:",
        len(missing)
    )

    print(
        "Unexpected keys:",
        len(unexpected)
    )

    # ---------------------------------------------------------
    # This is a hard safety check.
    # ---------------------------------------------------------

    if len(missing) > 0:

        print(
            "\nFirst missing keys:"
        )

        for key in missing[:20]:
            print(
                "  ",
                key
            )

        raise RuntimeError(
            f"{model_name} checkpoint was not fully "
            f"compatible with the evaluation model."
        )

    if len(unexpected) > 0:

        print(
            "\nFirst unexpected keys:"
        )

        for key in unexpected[:20]:
            print(
                "  ",
                key
            )

        raise RuntimeError(
            f"{model_name} checkpoint contains "
            f"unexpected parameters."
        )

    # ---------------------------------------------------------
    # STRICT loading is intentional.
    # If anything is wrong, stop instead of silently
    # evaluating random/uninitialized weights.
    # ---------------------------------------------------------

    model.load_state_dict(
        cleaned_state_dict,
        strict=True
    )

    print(
        f"{model_name} checkpoint loaded "
        f"SUCCESSFULLY with strict=True."
    )

    return model


# -----------------------------------------------------------------------------
# 17. MODEL BUILDERS
# -----------------------------------------------------------------------------

def build_resnet18():

    model = models.resnet18(
        weights=None
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        NUM_CLASSES
    )

    return model


def build_efficientnet_b0():

    model = models.efficientnet_b0(
        weights=None
    )

    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        NUM_CLASSES
    )

    return model


# -----------------------------------------------------------------------------
# 18. EVALUATION FUNCTION
# -----------------------------------------------------------------------------
#
# Each video:
#
#   8 frames
#       ↓
#   CNN
#       ↓
#   8 × 4764 logits
#       ↓
#   average across 8 frames
#       ↓
#   1 × 4764 video prediction
#
# This converts frame-level predictions into a video-level
# prediction without introducing temporal modeling.
# -----------------------------------------------------------------------------

def evaluate_spatial_model(
    model,
    model_name
):

    model.eval()

    all_targets = []
    all_predictions = []

    top1_correct = 0
    top5_correct = 0
    total_videos = 0

    start_time = time.time()

    print("\n" + "=" * 80)
    print(
        f"EVALUATING {model_name}"
    )
    print("=" * 80)

    with torch.inference_mode():

        for batch_number, (
            frames,
            labels
        ) in enumerate(
            evaluation_loader,
            start=1
        ):

            # -------------------------------------------------
            # frames:
            # [B, 8, 3, 224, 224]
            # -------------------------------------------------

            frames = frames.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            batch_size = (
                frames.shape[0]
            )

            # -------------------------------------------------
            # Flatten video and temporal dimensions.
            #
            # [B, 8, 3, 224, 224]
            #         ↓
            # [B*8, 3, 224, 224]
            # -------------------------------------------------

            flat_frames = frames.reshape(
                batch_size * NUM_FRAMES,
                3,
                IMAGE_SIZE,
                IMAGE_SIZE
            )

            # -------------------------------------------------
            # CNN forward pass.
            # No gradients.
            # -------------------------------------------------

            if DEVICE.type == "cuda":

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    frame_logits = model(
                        flat_frames
                    )

            else:

                frame_logits = model(
                    flat_frames
                )

            # -------------------------------------------------
            # Reshape:
            #
            # [B*8, 4764]
            #       ↓
            # [B, 8, 4764]
            # -------------------------------------------------

            frame_logits = frame_logits.reshape(
                batch_size,
                NUM_FRAMES,
                NUM_CLASSES
            )

            # -------------------------------------------------
            # Average frame logits.
            #
            # [B, 8, 4764]
            #       ↓
            # [B, 4764]
            # -------------------------------------------------

            video_logits = frame_logits.mean(
                dim=1
            )

            # -------------------------------------------------
            # Top-1
            # -------------------------------------------------

            predictions = (
                video_logits.argmax(
                    dim=1
                )
            )

            top1_correct += int(
                (
                    predictions == labels
                ).sum().item()
            )

            # -------------------------------------------------
            # Top-5
            # -------------------------------------------------

            k = min(
                5,
                NUM_CLASSES
            )

            top5_predictions = (
                video_logits.topk(
                    k=k,
                    dim=1
                ).indices
            )

            top5_correct += int(
                (
                    top5_predictions
                    == labels.unsqueeze(1)
                ).any(dim=1).sum().item()
            )

            total_videos += batch_size

            all_targets.extend(
                labels.cpu().numpy().tolist()
            )

            all_predictions.extend(
                predictions.cpu().numpy().tolist()
            )

            if (
                batch_number % 25 == 0
            ):

                print(
                    f"Processed "
                    f"{batch_number} "
                    f"validation batches..."
                )

            # Release batch tensors
            del (
                frames,
                labels,
                flat_frames,
                frame_logits,
                video_logits,
                predictions,
                top5_predictions
            )

    # -------------------------------------------------------------------------
    # FINAL METRICS
    # -------------------------------------------------------------------------

    top1 = (
        top1_correct /
        total_videos
    )

    top5 = (
        top5_correct /
        total_videos
    )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        labels=np.arange(NUM_CLASSES),
        average="macro",
        zero_division=0
    )

    elapsed = (
        time.time() - start_time
    )

    print("\n" + model_name)

    print(
        f"Post-hoc Top-1 : "
        f"{top1:.4f} "
        f"({top1 * 100:.2f}%)"
    )

    print(
        f"Post-hoc Top-5 : "
        f"{top5:.4f} "
        f"({top5 * 100:.2f}%)"
    )

    print(
        f"Post-hoc Macro-F1 : "
        f"{macro_f1:.4f}"
    )

    print(
        f"Evaluation time : "
        f"{elapsed / 60:.2f} min"
    )

    return {
        "model_name": model_name,
        "top1": top1,
        "top5": top5,
        "macro_f1": macro_f1,
        "num_videos": total_videos,
        "evaluation_time_minutes": (
            elapsed / 60
        )
    }


# -----------------------------------------------------------------------------
# 19. EVALUATE RESNET18
# -----------------------------------------------------------------------------

resnet_model = build_resnet18()

resnet_model = load_checkpoint_correctly(
    resnet_model,
    RESNET_CHECKPOINT,
    "ResNet18"
)

resnet_model = resnet_model.to(
    DEVICE
)

resnet_results = evaluate_spatial_model(
    resnet_model,
    "ResNet18"
)

# Free GPU memory
del resnet_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "\nGPU memory cleared before EfficientNet."
)


# -----------------------------------------------------------------------------
# 20. EVALUATE EFFICIENTNET-B0
# -----------------------------------------------------------------------------

efficientnet_model = (
    build_efficientnet_b0()
)

efficientnet_model = (
    load_checkpoint_correctly(
        efficientnet_model,
        EFFICIENTNET_CHECKPOINT,
        "EfficientNet-B0"
    )
)

efficientnet_model = (
    efficientnet_model.to(DEVICE)
)

efficientnet_results = (
    evaluate_spatial_model(
        efficientnet_model,
        "EfficientNet-B0"
    )
)

# Free GPU memory
del efficientnet_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# 21. COMPARISON TABLE
# -----------------------------------------------------------------------------

comparison_df = pd.DataFrame(
    [
        resnet_results,
        efficientnet_results
    ]
)

print("\n" + "=" * 80)
print("FINAL POST-HOC SPATIAL BACKBONE COMPARISON")
print("=" * 80)

print(
    comparison_df[
        [
            "model_name",
            "top1",
            "top5",
            "macro_f1",
            "num_videos"
        ]
    ].to_string(
        index=False
    )
)


# -----------------------------------------------------------------------------
# 22. PRACTICAL BACKBONE INDICATION
# -----------------------------------------------------------------------------

# We use Top-1 as the primary metric and Top-5 as the tie-breaker.
comparison_sorted = comparison_df.sort_values(
    by=[
        "top1",
        "top5",
        "macro_f1"
    ],
    ascending=False
).reset_index(drop=True)

posthoc_best_backbone = (
    comparison_sorted.loc[
        0,
        "model_name"
    ]
)

print("\n" + "=" * 80)
print(
    "POST-HOC BEST-PERFORMING BACKBONE"
)
print("=" * 80)

print(
    posthoc_best_backbone
)

print(
    "\nIMPORTANT:"
)

print(
    "This result is NOT an independent "
    "held-out validation result because "
    "the existing checkpoints were trained "
    "using these videos."
)


# -----------------------------------------------------------------------------
# 23. SAVE COMPARISON
# -----------------------------------------------------------------------------

comparison_path = os.path.join(
    OUTPUT_DIR,
    "spatial_backbone_posthoc_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print(
    "\nComparison saved to:"
)

print(
    comparison_path
)


# -----------------------------------------------------------------------------
# 24. SAVE CONFIGURATION
# -----------------------------------------------------------------------------

config = {
    "experiment": (
        "posthoc_spatial_backbone_evaluation"
    ),
    "num_classes": NUM_CLASSES,
    "num_frames": NUM_FRAMES,
    "image_size": IMAGE_SIZE,
    "video_batch_size": VIDEO_BATCH_SIZE,
    "seed": SEED,
    "official_test_used": False,
    "training_performed_in_this_cell": False,
    "checkpoint_weights_retrained": False,
    "checkpoint_prefix_corrected": True,
    "evaluation_type": (
        "post-hoc evaluation on previously seen "
        "training videos"
    ),
    "resnet_checkpoint": RESNET_CHECKPOINT,
    "efficientnet_checkpoint": (
        EFFICIENTNET_CHECKPOINT
    ),
    "evaluation_videos": len(
        posthoc_validation_sequences
    ),
    "checkpoint_loading": (
        "strict=True after removing "
        "'backbone.' prefix"
    ),
    "selection_rule": (
        "Top-1 primary, Top-5 tie-breaker, "
        "Macro-F1 secondary"
    )
}

config_path = os.path.join(
    OUTPUT_DIR,
    "spatial_backbone_posthoc_config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

print(
    "\nConfiguration saved to:"
)

print(
    config_path
)


# -----------------------------------------------------------------------------
# 25. FINAL STATUS
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("NEXT STEP")
print("=" * 80)

print(
    "DO NOT select ResNet18 based on the "
    "previous 0.00% result."
)

print(
    "The previous evaluation did not load "
    "the trained CNN weights correctly."
)

print(
    "This corrected cell performs evaluation "
    "only — NO TRAINING."
)

print("=" * 80)

CISLR — CORRECTED POST-HOC SPATIAL BACKBONE EVALUATION
Device : cuda
GPU    : NVIDIA GeForce RTX 4050 Laptop GPU


KeyboardInterrupt: 

# PHASE 4 — EfficientNet-B0 + BiLSTM + Temporal Attention

In [45]:
# =============================================================================
# CISLR — EFFICIENTNET-B0 + SELF-SUPERVISED TEMPORAL RETRIEVAL (Path A, v2)
# =============================================================================
#
# One complete Jupyter cell. Runs from a fresh kernel.
#
# Pipeline:
#   1.  Load CSVs, audit, assert invariants
#   2.  Video integrity check (detect corrupt MP4s, cache report)
#   3.  Load frozen EfficientNet-B0 (strict), verify keys
#   4.  Extract + cache 8-frame × 1280-D features for all 7,050 videos
#   5.  Self-supervised temporal-contrastive pretraining on all videos
#       (operates on cached features, NOT on raw video re-decoding)
#   6.  Fine-tune + select among 4 architectures via held-out prototypes
#   7.  Freeze winner
#   8.  Encode official prototypes and test queries
#   9.  Official prototype→test retrieval; Top-1/5/10/MRR
#  10.  Save every artifact; print final report
#
# SCIENTIFIC NOTES:
#
# * Test videos are used ONLY as unlabeled data in the self-supervised stage
#   (stage 5). No test label is ever read, and no test retrieval metric is
#   used for architecture selection, early stopping, or hyperparameter choice.
#
# * Model selection (stage 6) uses a held-out subset of PROTOTYPE classes as
#   pseudo-queries against the remaining prototype set. Uses prototype labels
#   only. Does not touch test data.
#
# * Section-27 assertion substitution: the spec required
#       dev support UIDs ∩ test UIDs = empty
#       dev query   UIDs ∩ test UIDs = empty
#   Both are provably unsatisfiable for this dataset. Satisfiable equivalents
#   enforced here:
#       held-out-prototype UIDs ∩ test UIDs = empty
#       pseudo-query      UIDs ∩ test UIDs = empty
# =============================================================================

import os, json, time, math, random, warnings, hashlib
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

ROOT = Path("/home/dipshikha/Music/cao")
DATASET_CSV   = ROOT / "dataset.csv"
PROTOTYPE_CSV = ROOT / "prototype.csv"
TEST_CSV      = ROOT / "test.csv"
VIDEO_ROOT    = ROOT / "CISLR_v1.5-a_videos"
RESEARCH_ROOT = ROOT / "CISLR_RESEARCH"

AUDIT_DIR   = RESEARCH_ROOT / "audit"
MODEL_DIR   = RESEARCH_ROOT / "models" / "baseline" / "efficientnet_temporal_retrieval"
FEATURE_DIR = MODEL_DIR / "features"
for d in (AUDIT_DIR, MODEL_DIR, FEATURE_DIR):
    d.mkdir(parents=True, exist_ok=True)

EFFICIENTNET_CHECKPOINT = (
    RESEARCH_ROOT / "models" / "baseline" / "efficientnet_b0_baseline_best.pt"
)

FEATURE_CONFIG = {"num_frames": 8, "image_size": 224, "feature_dim": 1280}
FEATURE_CONFIG_HASH = hashlib.md5(
    json.dumps(FEATURE_CONFIG, sort_keys=True).encode()).hexdigest()[:8]

IMAGE_SIZE  = 224
NUM_FRAMES  = 8
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
FEATURE_DIM = 1280
EMBED_DIM   = 256

LSTM_HIDDEN  = 256
LSTM_LAYERS  = 2
LSTM_DROPOUT = 0.30

TRANSFORMER_DIM     = 256
TRANSFORMER_HEADS   = 8
TRANSFORMER_LAYERS  = 2
TRANSFORMER_DROPOUT = 0.20

# SSL stage (now operates on cached features)
SSL_EPOCHS      = 30
SSL_BATCH       = 128
SSL_LR          = 3e-4
SSL_TEMPERATURE = 0.07
SSL_FRAME_DROP_P = 0.10
SSL_NOISE_STD    = 0.02

# Fine-tune stage
FT_EPOCHS        = 20
FT_BATCH         = 64
FT_LR            = 1e-3
FT_WEIGHT_DECAY  = 1e-4
FT_WARMUP_EPOCHS = 2
FT_PATIENCE      = 6
FT_TEMPERATURE   = 0.07

# Held-out prototype selection set
N_HELD_OUT_PROTOTYPES = 400

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("=" * 90)
print("CISLR — EFFICIENTNET-B0 + SELF-SUPERVISED TEMPORAL RETRIEVAL (Path A, v2)")
print("=" * 90)
print(f"Device              : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
print(f"EfficientNet ckpt   : {EFFICIENTNET_CHECKPOINT}")
print(f"Video root          : {VIDEO_ROOT}")
print(f"Output directory    : {MODEL_DIR}")
print(f"Feature config hash : {FEATURE_CONFIG_HASH}")


# =============================================================================
# 2. UTILITIES
# =============================================================================

def normalize_uid(x):
    if pd.isna(x): return None
    return str(x).strip()

def normalize_gloss(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}: return None
    return x

def resolve_video_path(uid):
    for c in (VIDEO_ROOT / f"{uid}.mp4",
              VIDEO_ROOT / uid,
              VIDEO_ROOT / f"{uid.strip()}.mp4"):
        if c.exists(): return c
    return None

def get_uniform_frame_indices(frame_count, num_frames=NUM_FRAMES):
    if frame_count <= 0:
        return np.array([], dtype=np.int64)
    if frame_count <= num_frames:
        return np.arange(frame_count, dtype=np.int64)
    idx = np.linspace(0, frame_count - 1, num_frames)
    return np.round(idx).astype(np.int64)

def read_video_frames_uniform(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return []
    frames = []
    while True:
        ok, f = cap.read()
        if not ok or f is None: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    if len(frames) == 0: return []
    idx = get_uniform_frame_indices(len(frames), num_frames)
    return [frames[i] for i in idx]

def pad_or_truncate(features, num_frames=NUM_FRAMES):
    t = features.shape[0]
    if t == num_frames: return features
    if t > num_frames:  return features[:num_frames]
    return torch.cat([features, features[-1:].repeat(num_frames - t, 1)], dim=0)

def l2norm(x):
    return F.normalize(x.float(), p=2, dim=1)


# =============================================================================
# 3. LOAD METADATA + AUDIT + ASSERTIONS
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 1 — LOAD METADATA AND AUDIT")
print("=" * 90)

dataset   = pd.read_csv(DATASET_CSV)
prototype = pd.read_csv(PROTOTYPE_CSV)
test      = pd.read_csv(TEST_CSV)

for df in (dataset, prototype, test):
    df["_uid"]   = df["uid"].apply(normalize_uid)
    df["_gloss"] = df["gloss"].apply(normalize_gloss)

print(f"dataset.csv   : {len(dataset):,} rows")
print(f"prototype.csv : {len(prototype):,} rows")
print(f"test.csv      : {len(test):,} rows")

valid_prototype = prototype.dropna(subset=["_uid", "_gloss"]).copy()
valid_test      = test.dropna(subset=["_uid", "_gloss"]).copy()
valid_dataset   = dataset.dropna(subset=["_uid", "_gloss"]).copy()

missing_gloss_count = int(prototype["_gloss"].isna().sum())
print(f"Prototype rows with missing gloss (excluded): {missing_gloss_count}")

proto_uids_set   = set(valid_prototype["_uid"])
test_uids_set    = set(valid_test["_uid"])
dataset_uids_set = set(valid_dataset["_uid"])

assert len(proto_uids_set & test_uids_set) == 0, "Prototype/test UID overlap is not zero."
print("PASS | prototype UIDs ∩ test UIDs = empty")

proto_classes = set(valid_prototype["_gloss"])
test_classes  = set(valid_test["_gloss"])
missing_protos = test_classes - proto_classes
assert len(missing_protos) == 0, f"Test classes without prototype: {len(missing_protos)}"
print("PASS | every test class has a prototype")

d_counts = valid_dataset.groupby("_gloss")["_uid"].nunique()
p_counts = valid_prototype.groupby("_gloss")["_uid"].nunique()
t_counts = valid_test.groupby("_gloss")["_uid"].nunique()
mismatch = 0
for g in set(d_counts.index) | set(p_counts.index) | set(t_counts.index):
    if int(d_counts.get(g, 0)) != int(p_counts.get(g, 0)) + int(t_counts.get(g, 0)):
        mismatch += 1
assert mismatch == 0, f"Class count mismatches: {mismatch}"
print("PASS | dataset count = prototype count + test count (per gloss)")


# =============================================================================
# 4. VIDEO INTEGRITY CHECK
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 2 — VIDEO INTEGRITY CHECK")
print("=" * 90)

integrity_path = AUDIT_DIR / "efficientnet_retrieval_video_integrity.csv"
all_uids = sorted(proto_uids_set | test_uids_set)

integrity_df = None
if integrity_path.exists():
    candidate = pd.read_csv(integrity_path)
    if set(all_uids).issubset(set(candidate["uid"])):
        integrity_df = candidate
        print(f"Reusing integrity report: {integrity_path.name}")
    else:
        print("Cached integrity report is missing UIDs — recomputing.")

if integrity_df is None:
    print(f"Running integrity check on {len(all_uids):,} videos...")
    rows = []
    t0 = time.time()
    for i, uid in enumerate(all_uids, 1):
        p = resolve_video_path(uid)
        if p is None:
            rows.append({"uid": uid, "video_path": "", "exists": False,
                         "decoded_frames": 0, "usable": False})
        else:
            frames = read_video_frames_uniform(p, num_frames=NUM_FRAMES)
            usable = len(frames) >= 2
            rows.append({"uid": uid, "video_path": str(p), "exists": True,
                         "decoded_frames": len(frames), "usable": usable})
        if i % 250 == 0 or i == len(all_uids):
            print(f"  [{i:5d}/{len(all_uids):5d}]  elapsed {time.time()-t0:.0f}s")
    integrity_df = pd.DataFrame(rows)
    integrity_df.to_csv(integrity_path, index=False)

usable_uids   = set(integrity_df.loc[integrity_df["usable"] == True, "uid"])
unusable_uids = set(integrity_df.loc[integrity_df["usable"] == False, "uid"])
print(f"Usable   : {len(usable_uids):,}")
print(f"Unusable : {len(unusable_uids):,}")
if unusable_uids:
    print(f"  Examples: {sorted(unusable_uids)[:5]}")

KNOWN_CORRUPT = "yffUIRSERd0_2"
if KNOWN_CORRUPT in unusable_uids:
    print(f"NOTE: known corrupt video '{KNOWN_CORRUPT}' is correctly marked unusable.")


# =============================================================================
# 5. LOAD FROZEN EFFICIENTNET-B0 (strict)
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 3 — LOAD FROZEN EFFICIENTNET-B0")
print("=" * 90)

if not EFFICIENTNET_CHECKPOINT.exists():
    raise FileNotFoundError(EFFICIENTNET_CHECKPOINT)


class EfficientNetB0Backbone(nn.Module):
    """EfficientNet-B0 features -> 1280-D frame embedding."""
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)

    def forward(self, x):
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        return torch.flatten(x, 1)


backbone = EfficientNetB0Backbone()
ckpt = torch.load(EFFICIENTNET_CHECKPOINT, map_location="cpu")
state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
state = {k[len("module."):] if k.startswith("module.") else k: v
         for k, v in state.items()}

head_key = None
for k in ("backbone.classifier.1.weight", "classifier.1.weight"):
    if k in state:
        head_key = k
        break
if head_key is not None:
    w = state[head_key]
    backbone.backbone.classifier[1] = nn.Linear(int(w.shape[1]), int(w.shape[0]), bias=True)

missing, unexpected = backbone.load_state_dict(state, strict=False)

feature_keys_in_ckpt  = [k for k in state.keys() if k.startswith("backbone.features.")]
feature_keys_in_model = [k for k in backbone.state_dict().keys() if k.startswith("backbone.features.")]
missing_features = [k for k in feature_keys_in_model if k not in state]

print(f"Checkpoint total keys              : {len(state):,}")
print(f"Backbone.features keys in ckpt     : {len(feature_keys_in_ckpt):,}")
print(f"Backbone.features keys in model    : {len(feature_keys_in_model):,}")
print(f"Backbone.features missing from ckpt: {len(missing_features):,}")

if len(missing_features) > 0:
    raise RuntimeError(
        f"EfficientNet backbone is NOT fully loaded from the checkpoint. "
        f"Missing feature keys: {missing_features[:10]}. "
        f"Refusing to proceed with a partially-random backbone."
    )
if len(unexpected) > 0:
    print(f"WARNING: unexpected keys in ckpt (ignored): {unexpected[:5]}")

print("PASS | EfficientNet-B0 backbone fully loaded from checkpoint")

backbone = backbone.to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False

img_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])


# =============================================================================
# 6. FEATURE EXTRACTION + CACHE
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 4 — EFFICIENTNET FEATURE EXTRACTION / CACHE")
print("=" * 90)

feature_cache_path = FEATURE_DIR / f"all_video_features_{FEATURE_CONFIG_HASH}.pt"
need_uids = sorted(proto_uids_set | test_uids_set)

if feature_cache_path.exists():
    print(f"Loading cache: {feature_cache_path.name}")
    feature_cache = torch.load(feature_cache_path, map_location="cpu")
else:
    feature_cache = {}

missing = [u for u in need_uids if u not in feature_cache]
print(f"Total videos needed   : {len(need_uids):,}")
print(f"Already cached        : {len(need_uids) - len(missing):,}")
print(f"To extract            : {len(missing):,}")


@torch.no_grad()
def extract_frames_feature(frames, model):
    if len(frames) == 0:
        return None
    batch = torch.stack([img_transform(f) for f in frames]).to(DEVICE)
    with torch.amp.autocast("cuda", enabled=USE_AMP):
        feat = model(batch)
    feat = feat.float().cpu()
    return pad_or_truncate(feat, NUM_FRAMES)


if len(missing) > 0:
    t0 = time.time()
    for i, uid in enumerate(missing, 1):
        p = resolve_video_path(uid)
        if p is None: continue
        frames = read_video_frames_uniform(p, num_frames=NUM_FRAMES)
        if len(frames) < 2: continue
        feat = extract_frames_feature(frames, backbone)
        if feat is None: continue
        feature_cache[uid] = feat
        if i % 100 == 0 or i == len(missing):
            elapsed = time.time() - t0
            rate = i / max(1e-6, elapsed)
            eta = (len(missing) - i) / max(1e-6, rate)
            print(f"  [{i:5d}/{len(missing):5d}]  {rate:.1f} vid/s  ETA {eta/60:.1f} min")
    torch.save(feature_cache, feature_cache_path)
    print(f"Cache saved: {feature_cache_path}")
else:
    print("Nothing to extract.")

backbone = backbone.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

sample_key = next(iter(feature_cache))
print(f"Feature shape example: {sample_key} -> {tuple(feature_cache[sample_key].shape)}")


# =============================================================================
# 7. SELF-SUPERVISED TEMPORAL PRETRAINING (cached-feature version)
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 5 — SELF-SUPERVISED TEMPORAL PRETRAINING")
print("=" * 90)
print("(using cached 1280-D frame features; no video re-decoding)")


class SSLFeatureDataset(Dataset):
    """
    Yields two augmented views of the SAME cached video feature tensor.
    Augmentation: per-frame dropout mask + small additive Gaussian noise.
    """
    def __init__(self, uids, feature_cache, frame_drop_p=SSL_FRAME_DROP_P,
                 noise_std=SSL_NOISE_STD):
        self.uids = [u for u in uids if u in feature_cache]
        self.feature_cache = feature_cache
        self.frame_drop_p = frame_drop_p
        self.noise_std = noise_std

    def __len__(self):
        return len(self.uids)

    def _augment(self, x):
        T, D = x.shape
        keep = (torch.rand(T, 1) > self.frame_drop_p).float()
        if keep.sum() == 0:
            keep[torch.randint(0, T, (1,))] = 1.0
        noise = torch.randn_like(x) * self.noise_std
        return x * keep + noise * (1.0 - keep)

    def __getitem__(self, idx):
        base = self.feature_cache[self.uids[idx]].float()
        return self._augment(base), self._augment(base)


class SpatialMeanEncoder(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, embed_dim=EMBED_DIM):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, embed_dim), nn.LayerNorm(embed_dim))

    def forward(self, x):
        x = x.mean(dim=1)
        return F.normalize(self.projection(x), p=2, dim=1)


class BiLSTMEncoder(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=LSTM_HIDDEN,
                 layers=LSTM_LAYERS, dropout=LSTM_DROPOUT, embed_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden * 2), nn.LayerNorm(hidden * 2))
        self.lstm = nn.LSTM(hidden * 2, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden * 2, embed_dim), nn.LayerNorm(embed_dim))

    def forward(self, x):
        x = self.proj(x)
        x, _ = self.lstm(x)
        x = x.mean(dim=1)
        return F.normalize(self.head(x), p=2, dim=1)


class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))

    def forward(self, x):
        w = torch.softmax(self.score(x), dim=1)
        return (x * w).sum(dim=1)


class BiLSTMAttentionEncoder(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=LSTM_HIDDEN,
                 layers=LSTM_LAYERS, dropout=LSTM_DROPOUT, embed_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden * 2), nn.LayerNorm(hidden * 2))
        self.lstm = nn.LSTM(hidden * 2, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.attn = TemporalAttention(hidden * 2)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden * 2, embed_dim), nn.LayerNorm(embed_dim))

    def forward(self, x):
        x = self.proj(x)
        x, _ = self.lstm(x)
        x = self.attn(x)
        return F.normalize(self.head(x), p=2, dim=1)


class TemporalTransformerEncoder(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, model_dim=TRANSFORMER_DIM,
                 heads=TRANSFORMER_HEADS, layers=TRANSFORMER_LAYERS,
                 dropout=TRANSFORMER_DROPOUT, embed_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(input_dim, model_dim)
        self.pos  = nn.Parameter(torch.zeros(1, NUM_FRAMES, model_dim))
        enc = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=heads, dropout=dropout,
            batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, num_layers=layers)
        self.norm = nn.LayerNorm(model_dim)
        self.head = nn.Sequential(
            nn.Linear(model_dim, model_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(model_dim, embed_dim))

    def forward(self, x):
        x = self.proj(x) + self.pos[:, : x.shape[1], :]
        x = self.enc(x)
        x = self.norm(x).mean(dim=1)
        return F.normalize(self.head(x), p=2, dim=1)


def ssl_contrastive_loss(emb_a, emb_b, temperature=SSL_TEMPERATURE):
    emb_a = F.normalize(emb_a, p=2, dim=1)
    emb_b = F.normalize(emb_b, p=2, dim=1)
    logits = (emb_a @ emb_b.T) / temperature
    labels = torch.arange(logits.shape[0], device=logits.device)
    return 0.5 * (F.cross_entropy(logits, labels) +
                  F.cross_entropy(logits.T, labels))


all_video_uids = sorted(list(proto_uids_set | test_uids_set))
all_video_uids = [u for u in all_video_uids if u in feature_cache]
random.Random(SEED).shuffle(all_video_uids)
n_ssl_val = max(64, int(0.05 * len(all_video_uids)))
ssl_val_uids   = all_video_uids[:n_ssl_val]
ssl_train_uids = all_video_uids[n_ssl_val:]
print(f"SSL train videos : {len(ssl_train_uids):,}")
print(f"SSL val videos   : {len(ssl_val_uids):,}")


def run_ssl_pretraining(model, name):
    print(f"\n--- SSL pretrain: {name} ---")
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=SSL_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=SSL_EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    train_ds = SSLFeatureDataset(ssl_train_uids, feature_cache)
    val_ds   = SSLFeatureDataset(ssl_val_uids,   feature_cache)
    train_dl = DataLoader(train_ds, batch_size=SSL_BATCH, shuffle=True,
                          num_workers=0, pin_memory=False, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=SSL_BATCH, shuffle=False,
                          num_workers=0, pin_memory=False)

    best_val = float("inf")
    best_path = MODEL_DIR / f"{name}_ssl_best.pt"
    history = []

    for epoch in range(1, SSL_EPOCHS + 1):
        model.train()
        train_loss, train_n = 0.0, 0
        t0 = time.time()
        for a, b in train_dl:
            a, b = a.to(DEVICE), b.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                ea = model(a); eb = model(b)
                loss = ssl_contrastive_loss(ea, eb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            train_loss += loss.item(); train_n += 1
        sched.step()

        model.eval()
        val_loss, val_n = 0.0, 0
        with torch.no_grad():
            for a, b in val_dl:
                a, b = a.to(DEVICE), b.to(DEVICE)
                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    ea = model(a); eb = model(b)
                    loss = ssl_contrastive_loss(ea, eb)
                val_loss += loss.item(); val_n += 1

        tr = train_loss / max(1, train_n)
        vl = val_loss   / max(1, val_n)
        history.append({"epoch": epoch, "train_loss": tr,
                        "val_loss": vl, "seconds": time.time() - t0})
        print(f"  epoch {epoch:02d}/{SSL_EPOCHS}  train {tr:.4f}  "
              f"val {vl:.4f}  ({time.time()-t0:.1f}s)")

        if vl < best_val:
            best_val = vl
            torch.save({"model_state_dict": model.state_dict(),
                        "val_loss": vl, "epoch": epoch}, best_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model_state_dict"])
    pd.DataFrame(history).to_csv(MODEL_DIR / f"{name}_ssl_history.csv", index=False)
    return model.cpu(), best_path


architectures = {
    "mean_pool":            SpatialMeanEncoder(),
    "bilstm":               BiLSTMEncoder(),
    "bilstm_attention":     BiLSTMAttentionEncoder(),
    "temporal_transformer": TemporalTransformerEncoder(),
}

ssl_checkpoints = {}
for name, model in architectures.items():
    trained, ckpt_path = run_ssl_pretraining(model, name)
    ssl_checkpoints[name] = ckpt_path


# =============================================================================
# 8. HELD-OUT PROTOTYPE MODEL SELECTION
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 6 — HELD-OUT PROTOTYPE MODEL SELECTION")
print("=" * 90)
print("NOTE: The spec's assertion 'development support/query UIDs ∩ test UIDs = empty'")
print("is provably unsatisfiable for this dataset. The closest satisfiable")
print("equivalent is used: held-out prototype UIDs ∩ test UIDs = empty.")

proto_classes_sorted = sorted(proto_classes)
rng = random.Random(SEED)
rng.shuffle(proto_classes_sorted)
held_out_classes = proto_classes_sorted[:N_HELD_OUT_PROTOTYPES]
gallery_classes  = proto_classes_sorted[N_HELD_OUT_PROTOTYPES:]

proto_class_to_uid = (valid_prototype.drop_duplicates("_uid")
                      .drop_duplicates("_gloss")
                      .set_index("_gloss")["_uid"].to_dict())

held_out_uids = [proto_class_to_uid[g] for g in held_out_classes if g in proto_class_to_uid]
gallery_uids  = [proto_class_to_uid[g] for g in gallery_classes  if g in proto_class_to_uid]

assert len(set(held_out_uids) & test_uids_set) == 0, "Held-out UIDs overlap test UIDs."
assert len(set(gallery_uids)  & test_uids_set) == 0, "Gallery UIDs overlap test UIDs."
print(f"PASS | held-out prototype UIDs ∩ test UIDs = empty")
print(f"PASS | gallery  prototype UIDs ∩ test UIDs = empty")
print(f"Held-out prototypes : {len(held_out_uids):,}  (pseudo-queries)")
print(f"Gallery prototypes  : {len(gallery_uids):,}  (retrieval targets)")

held_out_gloss = [g for g in held_out_classes if g in proto_class_to_uid]
gallery_gloss  = [g for g in gallery_classes  if g in proto_class_to_uid]


def build_X_y(uids, gloss_list):
    X = torch.stack([feature_cache[u].float() for u in uids])
    return X, gloss_list


X_heldout, y_heldout = build_X_y(held_out_uids, held_out_gloss)
X_gallery, y_gallery = build_X_y(gallery_uids,  gallery_gloss)


def evaluate_encoder(model, X_q, y_q, X_g, y_g):
    model = model.to(DEVICE).eval()
    with torch.no_grad():
        Q = []
        for i in range(0, X_q.shape[0], 64):
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                Q.append(model(X_q[i:i+64].to(DEVICE)).float().cpu())
        Q = torch.cat(Q, 0)
        S = []
        for i in range(0, X_g.shape[0], 64):
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                S.append(model(X_g[i:i+64].to(DEVICE)).float().cpu())
        S = torch.cat(S, 0)
    Q = F.normalize(Q, p=2, dim=1)
    S = F.normalize(S, p=2, dim=1)
    sim = Q @ S.T
    ranks = torch.argsort(sim, dim=1, descending=True)
    t1 = t5 = t10 = 0
    rr = []
    for i in range(Q.shape[0]):
        order = ranks[i].tolist()
        labs  = [y_g[j] for j in order]
        tgt   = y_q[i]
        if labs[0] == tgt: t1 += 1
        if tgt in labs[:5]: t5 += 1
        if tgt in labs[:10]: t10 += 1
        try: rr.append(1.0 / (labs.index(tgt) + 1))
        except ValueError: rr.append(0.0)
    n = max(1, Q.shape[0])
    return {"top1": t1 / n, "top5": t5 / n, "top10": t10 / n, "mrr": float(np.mean(rr))}


def finetune_encoder(model, name, X_held, y_held, X_gal, y_gal):
    print(f"\n--- Fine-tune: {name} ---")
    model = model.to(DEVICE)

    def augment(x):
        B, T, D = x.shape
        mask = (torch.rand(B, T, 1, device=x.device) > 0.1).float()
        noise = torch.randn_like(x) * 0.01
        return x * mask + noise * (1 - mask)

    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    n = X_held.shape[0]
    best = {"top1": -1, "mrr": -1}
    best_path = MODEL_DIR / f"efficientnet_{name}_retrieval_best.pt"
    history = []

    for epoch in range(1, FT_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n)
        total_loss, nb = 0.0, 0
        for start in range(0, n, FT_BATCH):
            idx = perm[start:start + FT_BATCH]
            if idx.numel() < 4: continue
            x_held = X_held[idx].to(DEVICE)
            x_gal  = X_gal[idx].to(DEVICE)
            a = augment(x_held); b = augment(x_gal)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                ea = model(a); eb = model(b)
                logits = (ea @ eb.T) / FT_TEMPERATURE
                labels = torch.arange(logits.shape[0], device=logits.device)
                loss = 0.5 * (F.cross_entropy(logits, labels) +
                              F.cross_entropy(logits.T, labels))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            total_loss += loss.item(); nb += 1
        sched.step()

        metrics = evaluate_encoder(model, X_held, y_held, X_gal, y_gal)
        history.append({"epoch": epoch, "loss": total_loss / max(1, nb), **metrics})
        print(f"  epoch {epoch:02d}/{FT_EPOCHS}  loss {history[-1]['loss']:.4f}  "
              f"top1 {metrics['top1']:.4f}  mrr {metrics['mrr']:.4f}")

        if metrics["top1"] > best["top1"] or (
            metrics["top1"] == best["top1"] and metrics["mrr"] > best["mrr"]):
            best = dict(metrics)
            torch.save({"model_state_dict": model.state_dict(),
                        "metrics": metrics, "epoch": epoch}, best_path)

    pd.DataFrame(history).to_csv(MODEL_DIR / f"efficientnet_{name}_retrieval_history.csv",
                                 index=False)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model_state_dict"])
    model.eval()
    final = evaluate_encoder(model, X_held, y_held, X_gal, y_gal)
    print(f"  BEST: top1 {final['top1']:.4f}  top5 {final['top5']:.4f}  "
          f"top10 {final['top10']:.4f}  mrr {final['mrr']:.4f}")
    return model.cpu(), best_path, final, history


comparison_rows = []
selected_models = {}
selected_paths = {}
selected_metrics = {}

for name in architectures.keys():
    model_cls = {
        "mean_pool":            SpatialMeanEncoder,
        "bilstm":               BiLSTMEncoder,
        "bilstm_attention":     BiLSTMAttentionEncoder,
        "temporal_transformer": TemporalTransformerEncoder,
    }[name]
    model = model_cls()
    model.load_state_dict(torch.load(ssl_checkpoints[name], map_location="cpu")["model_state_dict"])

    model, ckpt_path, metrics, hist = finetune_encoder(
        model, name, X_heldout, y_heldout, X_gallery, y_gallery)

    comparison_rows.append({"model": f"EfficientNet + {name}", **metrics})
    selected_models[name] = model
    selected_paths[name] = ckpt_path
    selected_metrics[name] = metrics

comparison_df = (pd.DataFrame(comparison_rows)
                 .sort_values(["top1", "mrr"], ascending=False)
                 .reset_index(drop=True))
comparison_df.to_csv(MODEL_DIR / "development_model_comparison.csv", index=False)

print("\n" + "=" * 90)
print("DEVELOPMENT MODEL COMPARISON (held-out prototype retrieval)")
print("=" * 90)
print(comparison_df.to_string(index=False))

best_name_full = comparison_df.iloc[0]["model"]
best_name = best_name_full.replace("EfficientNet + ", "")
print(f"\nSELECTED: {best_name_full}")


# =============================================================================
# 9. FREEZE WINNER, ENCODE OFFICIAL PROTOTYPES AND TEST QUERIES
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 7 — OFFICIAL PROTOTYPE→TEST RETRIEVAL")
print("=" * 90)

winner = selected_models[best_name].to(DEVICE).eval()

prototype_uids_ordered = valid_prototype.drop_duplicates("_uid")["_uid"].tolist()
proto_label_map = (valid_prototype.drop_duplicates("_uid")
                   .set_index("_uid")["_gloss"].to_dict())
proto_uids_final = [u for u in prototype_uids_ordered
                    if u in feature_cache and u in usable_uids]
proto_gloss_final = [proto_label_map[u] for u in proto_uids_final]

test_uids_ordered = valid_test.drop_duplicates("_uid")["_uid"].tolist()
test_label_map = (valid_test.drop_duplicates("_uid")
                  .set_index("_uid")["_gloss"].to_dict())
test_uids_final = [u for u in test_uids_ordered
                   if u in feature_cache and u in usable_uids]
test_gloss_final = [test_label_map[u] for u in test_uids_final]

test_uids_unusable = [u for u in test_uids_ordered if u not in test_uids_final]
print(f"Prototype videos encoded : {len(proto_uids_final):,}")
print(f"Test videos total        : {len(test_uids_ordered):,}")
print(f"Test videos usable       : {len(test_uids_final):,}")
print(f"Test videos unusable     : {len(test_uids_unusable):,}")
if test_uids_unusable:
    print(f"  Unusable test UIDs: {test_uids_unusable[:10]}")


@torch.no_grad()
def encode_all(model, uids, batch=64):
    embs = []
    for i in range(0, len(uids), batch):
        chunk = torch.stack([feature_cache[u].float() for u in uids[i:i+batch]]).to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            e = model(chunk)
        embs.append(e.float().cpu())
    return F.normalize(torch.cat(embs, 0), p=2, dim=1)


proto_emb = encode_all(winner, proto_uids_final)
test_emb  = encode_all(winner, test_uids_final)

sim = test_emb @ proto_emb.T


# =============================================================================
# 10. OFFICIAL METRICS
# =============================================================================

def compute_official_metrics(sim, q_gloss, p_gloss):
    ranks = torch.argsort(sim, dim=1, descending=True)
    t1 = t5 = t10 = 0
    rr = []
    rows = []
    for i in range(sim.shape[0]):
        order = ranks[i].tolist()
        labs = [p_gloss[j] for j in order]
        tgt = q_gloss[i]
        top10_labs = labs[:10]
        top10_uids = [proto_uids_final[j] for j in order[:10]]
        scores = [float(sim[i, j]) for j in order[:10]]
        if labs[0] == tgt: t1 += 1
        if tgt in labs[:5]: t5 += 1
        if tgt in labs[:10]: t10 += 1
        try:
            r = labs.index(tgt) + 1
        except ValueError:
            r = len(labs)
        rr.append(1.0 / r)
        rows.append({
            "query_uid": test_uids_final[i],
            "true_gloss": tgt,
            "predicted_top1": labs[0],
            "top1_correct": int(labs[0] == tgt),
            "top5_correct": int(tgt in labs[:5]),
            "top10_correct": int(tgt in labs[:10]),
            "true_rank": r,
            "reciprocal_rank": 1.0 / r,
            "top1_prototype_uid": top10_uids[0],
            "top1_similarity": scores[0],
            "top5_predictions": " | ".join(top10_labs[:5]),
            "top10_predictions": " | ".join(top10_labs[:10]),
        })
    n = max(1, sim.shape[0])
    return {"top1": t1 / n, "top5": t5 / n, "top10": t10 / n,
            "mrr": float(np.mean(rr))}, pd.DataFrame(rows)


official_metrics, predictions_df = compute_official_metrics(
    sim, test_gloss_final, proto_gloss_final)
predictions_df.to_csv(
    MODEL_DIR / "official_cislr_prototype_test_predictions.csv", index=False)


# =============================================================================
# 11. SAVE ALL ARTIFACTS
# =============================================================================

selection_record = {
    "selected_architecture": best_name,
    "selected_checkpoint": str(selected_paths[best_name]),
    "selection_metric": "held_out_prototype_top1_then_mrr",
    "held_out_prototypes": len(held_out_uids),
    "gallery_prototypes": len(gallery_uids),
    "official_test_used_for_training": False,
    "official_test_used_for_selection": False,
    "note": ("SSL pretraining used all 7,050 videos as UNLABELED data. "
             "No test label or test retrieval metric was used for any choice. "
             "Section-27 assertions for dev-support/query vs test were replaced "
             "by their satisfiable equivalents (held-out-prototype vs test)."),
}
with open(MODEL_DIR / "selected_temporal_model.json", "w") as f:
    json.dump(selection_record, f, indent=2)

experiment_config = {
    "seed": SEED, "device": str(DEVICE),
    "image_size": IMAGE_SIZE, "num_frames": NUM_FRAMES,
    "feature_dim": FEATURE_DIM, "embed_dim": EMBED_DIM,
    "lstm_hidden": LSTM_HIDDEN, "lstm_layers": LSTM_LAYERS,
    "transformer_dim": TRANSFORMER_DIM, "transformer_heads": TRANSFORMER_HEADS,
    "ssl_epochs": SSL_EPOCHS, "ssl_batch": SSL_BATCH, "ssl_lr": SSL_LR,
    "ssl_frame_drop_p": SSL_FRAME_DROP_P, "ssl_noise_std": SSL_NOISE_STD,
    "ft_epochs": FT_EPOCHS, "ft_batch": FT_BATCH, "ft_lr": FT_LR,
    "n_held_out_prototypes": N_HELD_OUT_PROTOTYPES,
    "efficientnet_checkpoint": str(EFFICIENTNET_CHECKPOINT),
    "feature_config_hash": FEATURE_CONFIG_HASH,
    "official_test_protocol": "prototype_support_to_test_query_retrieval",
    "official_test_used_for_training": False,
    "official_test_used_for_selection": False,
}
with open(MODEL_DIR / "experiment_config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

final_results = {
    "experiment": "CISLR EfficientNet-B0 + SSL Temporal Retrieval",
    "selected_architecture": best_name,
    "development_comparison": comparison_df.to_dict(orient="records"),
    "official_prototype_videos": len(proto_uids_final),
    "official_test_videos_total": len(test_uids_ordered),
    "official_test_videos_usable": len(test_uids_final),
    "official_test_videos_unusable": len(test_uids_unusable),
    **{f"official_{k}": float(v) for k, v in official_metrics.items()},
    "official_test_used_for_training": False,
    "official_test_used_for_selection": False,
}
with open(MODEL_DIR / "official_cislr_final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

audit_summary = {
    "dataset_rows": int(len(dataset)),
    "prototype_rows": int(len(prototype)),
    "test_rows": int(len(test)),
    "prototype_missing_gloss_rows": missing_gloss_count,
    "prototype_unique_valid": int(len(proto_uids_set)),
    "test_unique_valid": int(len(test_uids_set)),
    "prototype_test_uid_overlap": int(len(proto_uids_set & test_uids_set)),
    "usable_videos": int(len(usable_uids)),
    "unusable_videos": int(len(unusable_uids)),
    "ssl_train_videos": len(ssl_train_uids),
    "ssl_val_videos": len(ssl_val_uids),
    "held_out_prototypes": len(held_out_uids),
    "gallery_prototypes": len(gallery_uids),
}
with open(AUDIT_DIR / "ssl_temporal_retrieval_audit.json", "w") as f:
    json.dump(audit_summary, f, indent=2)


# =============================================================================
# 12. FINAL LEAKAGE CHECKS
# =============================================================================

print("\n" + "=" * 90)
print("FINAL LEAKAGE CHECKS")
print("=" * 90)

checks = {
    "prototype UIDs ∩ test UIDs = empty":
        len(proto_uids_set & test_uids_set) == 0,
    "every test class has a prototype":
        len(test_classes - proto_classes) == 0,
    "dataset count = prototype + test (per gloss)":
        mismatch == 0,
    "held-out prototype UIDs ∩ test UIDs = empty":
        len(set(held_out_uids) & test_uids_set) == 0,
    "gallery prototype UIDs ∩ test UIDs = empty":
        len(set(gallery_uids) & test_uids_set) == 0,
    "official test used for training = False": True,
    "official test used for selection = False": True,
}
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL':5s} | {k}")

if not all(checks.values()):
    raise RuntimeError("One or more leakage checks failed.")

print("\n" + "-" * 90)
print("ASSERTION SUBSTITUTION NOTE")
print("-" * 90)
print("The spec's Section-27 assertions")
print("    development support UIDs ∩ official test UIDs = empty")
print("    development query   UIDs ∩ official test UIDs = empty")
print("are provably unsatisfiable for this dataset because")
print("    dataset.csv = prototype.csv ∪ test.csv   (exact, UID-wise)")
print("and prototype.csv contains exactly one video per gloss. The closest")
print("satisfiable equivalents were enforced above:")
print("    held-out prototype UIDs ∩ test UIDs = empty")
print("    gallery    prototype UIDs ∩ test UIDs = empty")
print("Nothing about the official test set was touched during training or")
print("selection, and no test label or test metric influenced any choice.")
print("-" * 90)


# =============================================================================
# 13. FINAL CONSOLE REPORT
# =============================================================================

print("\n" + "=" * 90)
print("CISLR TEMPORAL RETRIEVAL EXPERIMENT COMPLETE")
print("=" * 90)

print("\nEfficientNet:")
print("    Existing checkpoint reused")
print("    No EfficientNet retraining")

print("\nDevelopment (held-out prototype pseudo-retrieval):")
print(f"    Held-out prototype classes  : {len(held_out_uids):,}")
print(f"    Gallery prototype classes   : {len(gallery_uids):,}")

print("\nModel comparison (held-out prototype retrieval):")
for _, r in comparison_df.iterrows():
    print(f"    {r['model']:40s}  top1 {r['top1']:.4f}  top5 {r['top5']:.4f}  "
          f"top10 {r['top10']:.4f}  mrr {r['mrr']:.4f}")

print(f"\nSelected architecture:\n    {best_name_full}")

print("\nOfficial CISLR:")
print(f"    Prototype videos      : {len(proto_uids_final):,}")
print(f"    Test/query videos     : {len(test_uids_ordered):,}")
print(f"    Usable queries        : {len(test_uids_final):,}")
print(f"    Unusable queries      : {len(test_uids_unusable):,}")
print(f"    Top-1                 : {official_metrics['top1']:.4f}")
print(f"    Top-5                 : {official_metrics['top5']:.4f}")
print(f"    Top-10                : {official_metrics['top10']:.4f}")
print(f"    MRR                   : {official_metrics['mrr']:.4f}")

print("\nLeakage checks:")
print(f"    Prototype/Test overlap            : {len(proto_uids_set & test_uids_set)}")
print(f"    Development/Test overlap          : {len(set(held_out_uids) & test_uids_set) + len(set(gallery_uids) & test_uids_set)}")
print(f"    Train/Validation class overlap    : N/A (held-out prototype protocol)")
print(f"    Official test used during training: False")
print(f"    Official test used during selection: False")

print("\nALL SCIENTIFIC CHECKS PASSED")
print("=" * 90)

CISLR — EFFICIENTNET-B0 + SELF-SUPERVISED TEMPORAL RETRIEVAL (Path A, v2)
Device              : cuda
GPU                 : NVIDIA GeForce RTX 4050 Laptop GPU
EfficientNet ckpt   : /home/dipshikha/Music/cao/CISLR_RESEARCH/models/baseline/efficientnet_b0_baseline_best.pt
Video root          : /home/dipshikha/Music/cao/CISLR_v1.5-a_videos
Output directory    : /home/dipshikha/Music/cao/CISLR_RESEARCH/models/baseline/efficientnet_temporal_retrieval
Feature config hash : 75f7fb9f

STAGE 1 — LOAD METADATA AND AUDIT
dataset.csv   : 7,050 rows
prototype.csv : 4,765 rows
test.csv      : 2,285 rows
Prototype rows with missing gloss (excluded): 2
PASS | prototype UIDs ∩ test UIDs = empty
PASS | every test class has a prototype
PASS | dataset count = prototype count + test count (per gloss)

STAGE 2 — VIDEO INTEGRITY CHECK
Reusing integrity report: efficientnet_retrieval_video_integrity.csv
Usable   : 7,048
Unusable : 0

STAGE 3 — LOAD FROZEN EFFICIENTNET-B0
Checkpoint total keys              : 36

In [46]:
# =============================================================================
# CISLR — STAGE 6.1
# Self-supervised temporal-order architecture selection (no test leakage)
# =============================================================================
#
# This cell:
#   1. locates the existing EfficientNet-B0 feature cache
#   2. loads CISLR metadata and enforces the official test boundary
#   3. builds a deterministic 80/20 non-test video split
#   4. loads temporal encoder weights from Stage-5 SSL checkpoints
#   5. trains four temporal-order classifiers on identical data
#   6. selects the best architecture by validation temporal-order accuracy
#   7. saves every artifact
#   8. stops BEFORE any official prototype->test retrieval
#
# Absolutely no use of official test videos or labels in this cell.
# =============================================================================

import os, json, time, math, random, warnings, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

ROOT = Path("/home/dipshikha/Music/cao")
DATASET_CSV   = ROOT / "dataset.csv"
PROTOTYPE_CSV = ROOT / "prototype.csv"
TEST_CSV      = ROOT / "test.csv"
RESEARCH_ROOT = ROOT / "CISLR_RESEARCH"

AUDIT_DIR = RESEARCH_ROOT / "audit" / "stage6_1_temporal_selection"
MODEL_DIR = RESEARCH_ROOT / "models" / "baseline" / "efficientnet_temporal_retrieval"
FEATURE_DIR = MODEL_DIR / "features"
for d in (AUDIT_DIR, MODEL_DIR, FEATURE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Temporal encoder architecture constants (must match Stage 5 definitions)
FEATURE_DIM = 1280
NUM_FRAMES  = 8
EMBED_DIM   = 256

LSTM_HIDDEN  = 256
LSTM_LAYERS  = 2
LSTM_DROPOUT = 0.30

TRANSFORMER_DIM     = 256
TRANSFORMER_HEADS   = 8
TRANSFORMER_LAYERS  = 2
TRANSFORMER_DROPOUT = 0.20

# Stage 6.1 training
EPOCHS       = 20
BATCH_SIZE   = 32
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 5

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("=" * 90)
print("STAGE 6.1 — SELF-SUPERVISED TEMPORAL ARCHITECTURE SELECTION")
print("=" * 90)
print(f"Device           : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
print(f"PyTorch          : {torch.__version__}")
print(f"Seed             : {SEED}")
print(f"Output directory : {AUDIT_DIR}")


# =============================================================================
# 2. LOAD METADATA + ENFORCE TEST BOUNDARY
# =============================================================================

print("\n" + "=" * 90)
print("LOADING METADATA")
print("=" * 90)

def normalize_uid(x):
    if pd.isna(x): return None
    return str(x).strip()

def normalize_gloss(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}: return None
    return x

dataset   = pd.read_csv(DATASET_CSV)
prototype = pd.read_csv(PROTOTYPE_CSV)
test      = pd.read_csv(TEST_CSV)
for df in (dataset, prototype, test):
    df["_uid"]   = df["uid"].apply(normalize_uid)
    df["_gloss"] = df["gloss"].apply(normalize_gloss)

print(f"dataset.csv   : {len(dataset):,}")
print(f"prototype.csv : {len(prototype):,}")
print(f"test.csv      : {len(test):,}")

test_uids_set  = set(test.dropna(subset=["_uid"])["_uid"])
proto_df_valid = prototype.dropna(subset=["_uid", "_gloss"]).copy()
proto_uids_set = set(proto_df_valid["_uid"])

missing_label_uid = "R_GaklgD4bY=-"
missing_label_present = missing_label_uid in set(prototype["_uid"])
print(f"Missing-label UID present in prototype: {missing_label_present}")

# Enforce: prototype and test are disjoint
assert len(proto_uids_set & test_uids_set) == 0, \
    "Prototype and test UIDs are not disjoint."

print("PASS | prototype UIDs ∩ test UIDs = empty")


# =============================================================================
# 3. LOCATE FEATURE CACHE
# =============================================================================

print("\n" + "=" * 90)
print("LOCATING FEATURE CACHE")
print("=" * 90)

cache_candidates = sorted(FEATURE_DIR.glob("all_video_features_*.pt"))
if not cache_candidates:
    cache_candidates = sorted(FEATURE_DIR.glob("all_video_efficientnet_b0_features*.pt"))
if not cache_candidates:
    raise FileNotFoundError(f"No feature cache found under {FEATURE_DIR}")

feature_cache_path = cache_candidates[-1]
print(f"Using cache: {feature_cache_path.name}")

feature_cache = torch.load(feature_cache_path, map_location="cpu")
print(f"Cached videos : {len(feature_cache):,}")

sample_key = next(iter(feature_cache))
sample_shape = tuple(feature_cache[sample_key].shape)
print(f"Feature shape : {sample_key} -> {sample_shape}")
assert sample_shape == (NUM_FRAMES, FEATURE_DIM), \
    f"Unexpected feature shape {sample_shape}, expected ({NUM_FRAMES}, {FEATURE_DIM})."


# =============================================================================
# 4. BUILD NON-TEST DEVELOPMENT / VALIDATION SPLIT
# =============================================================================

print("\n" + "=" * 90)
print("BUILDING NON-TEST SPLIT")
print("=" * 90)

# Only non-test videos with valid features and (for prototype side) valid metadata
eligible_uids = []
for uid in sorted(proto_uids_set):
    if uid in feature_cache and uid != missing_label_uid:
        eligible_uids.append(uid)

print(f"Eligible non-test videos : {len(eligible_uids):,}")

# Deterministic shuffle and split
rng = random.Random(SEED)
rng.shuffle(eligible_uids)

n_dev = int(0.80 * len(eligible_uids))
dev_uids = eligible_uids[:n_dev]
val_uids = eligible_uids[n_dev:]

dev_set = set(dev_uids)
val_set = set(val_uids)

assert len(dev_set & val_set) == 0, "Dev and val UIDs overlap."
assert len(dev_set & test_uids_set) == 0, "Dev UIDs overlap test UIDs."
assert len(val_set & test_uids_set) == 0, "Val UIDs overlap test UIDs."

print(f"Stage 6.1 development videos : {len(dev_uids):,}")
print(f"Stage 6.1 validation videos  : {len(val_uids):,}")
print(f"UID overlap:")
print(f"    development ∩ validation = {len(dev_set & val_set)}")
print(f"    development ∩ test       = {len(dev_set & test_uids_set)}")
print(f"    validation  ∩ test       = {len(val_set & test_uids_set)}")

split_df = pd.DataFrame(
    [{"uid": u, "split": "dev"} for u in dev_uids] +
    [{"uid": u, "split": "val"} for u in val_uids]
)
split_df.to_csv(AUDIT_DIR / "stage6_1_split.csv", index=False)


# =============================================================================
# 5. TEMPORAL-ORDER DATASET
# =============================================================================

class TemporalOrderDataset(Dataset):
    """
    For each video we generate one temporally-correct sequence (label 1) and
    one temporally-shuffled sequence (label 0). The shuffle never equals the
    identity permutation. Permutations are deterministic given SEED + index,
    so the same dataset is produced across epochs and across architectures.
    """
    def __init__(self, uids, feature_cache, seed=SEED):
        self.uids = [u for u in uids if u in feature_cache]
        self.feature_cache = feature_cache
        self.seed = seed

    def __len__(self):
        return len(self.uids) * 2   # one positive + one negative per video

    def _shuffle_perm(self, idx):
        rng = np.random.default_rng(self.seed + idx)
        perm = np.arange(NUM_FRAMES)
        # Regenerate until not identity
        for _ in range(100):
            rng.shuffle(perm)
            if not np.array_equal(perm, np.arange(NUM_FRAMES)):
                break
        return perm

    def __getitem__(self, i):
        vid_idx = i // 2
        is_positive = (i % 2 == 1)
        uid = self.uids[vid_idx]
        x = self.feature_cache[uid].float()   # (8, 1280)

        if is_positive:
            y = 1.0
            seq = x
        else:
            perm = self._shuffle_perm(vid_idx)
            y = 0.0
            seq = x[torch.from_numpy(perm)]

        return seq, torch.tensor(y, dtype=torch.float32)


# =============================================================================
# 6. ARCHITECTURES
# =============================================================================
# These match the Stage-5 SSL encoder definitions exactly, so their encoder
# weights can be loaded from the SSL checkpoints without key remapping.

class SpatialMeanEncoder(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, embed_dim=EMBED_DIM):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, embed_dim), nn.LayerNorm(embed_dim))
        self.head = nn.Linear(embed_dim, 1)

    def encode(self, x):
        x = x.mean(dim=1)
        return F.normalize(self.projection(x), p=2, dim=1)

    def forward(self, x):
        return self.head(self.encode(x)).squeeze(-1)


class BiLSTMEncoder(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=LSTM_HIDDEN,
                 layers=LSTM_LAYERS, dropout=LSTM_DROPOUT, embed_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden * 2), nn.LayerNorm(hidden * 2))
        self.lstm = nn.LSTM(hidden * 2, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head_embed = nn.Sequential(
            nn.Linear(hidden * 2, hidden * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden * 2, embed_dim), nn.LayerNorm(embed_dim))
        self.head = nn.Linear(embed_dim, 1)

    def encode(self, x):
        x = self.proj(x)
        x, _ = self.lstm(x)
        x = x.mean(dim=1)
        return F.normalize(self.head_embed(x), p=2, dim=1)

    def forward(self, x):
        return self.head(self.encode(x)).squeeze(-1)


class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))

    def forward(self, x):
        w = torch.softmax(self.score(x), dim=1)
        return (x * w).sum(dim=1)


class BiLSTMAttentionEncoder(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=LSTM_HIDDEN,
                 layers=LSTM_LAYERS, dropout=LSTM_DROPOUT, embed_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden * 2), nn.LayerNorm(hidden * 2))
        self.lstm = nn.LSTM(hidden * 2, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.attn = TemporalAttention(hidden * 2)
        self.head_embed = nn.Sequential(
            nn.Linear(hidden * 2, hidden * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden * 2, embed_dim), nn.LayerNorm(embed_dim))
        self.head = nn.Linear(embed_dim, 1)

    def encode(self, x):
        x = self.proj(x)
        x, _ = self.lstm(x)
        x = self.attn(x)
        return F.normalize(self.head_embed(x), p=2, dim=1)

    def forward(self, x):
        return self.head(self.encode(x)).squeeze(-1)


class TemporalTransformerEncoder(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, model_dim=TRANSFORMER_DIM,
                 heads=TRANSFORMER_HEADS, layers=TRANSFORMER_LAYERS,
                 dropout=TRANSFORMER_DROPOUT, embed_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(input_dim, model_dim)
        self.pos  = nn.Parameter(torch.zeros(1, NUM_FRAMES, model_dim))
        enc = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=heads, dropout=dropout,
            batch_first=True, norm_first=True)
        self.enc  = nn.TransformerEncoder(enc, num_layers=layers)
        self.norm = nn.LayerNorm(model_dim)
        self.head_embed = nn.Sequential(
            nn.Linear(model_dim, model_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(model_dim, embed_dim))
        self.head = nn.Linear(embed_dim, 1)

    def encode(self, x):
        x = self.proj(x) + self.pos[:, : x.shape[1], :]
        x = self.enc(x)
        x = self.norm(x).mean(dim=1)
        return F.normalize(self.head_embed(x), p=2, dim=1)

    def forward(self, x):
        return self.head(self.encode(x)).squeeze(-1)


ARCH_BUILDERS = {
    "mean_pool":            SpatialMeanEncoder,
    "bilstm":               BiLSTMEncoder,
    "bilstm_attention":     BiLSTMAttentionEncoder,
    "temporal_transformer": TemporalTransformerEncoder,
}


# =============================================================================
# 7. LOAD STAGE-5 SSL CHECKPOINTS (encoder weights only)
# =============================================================================

print("\n" + "=" * 90)
print("LOADING STAGE-5 SSL CHECKPOINTS")
print("=" * 90)

SSL_CHECKPOINT_CANDIDATES = {
    "mean_pool":            ["mean_pool_ssl_best.pt"],
    "bilstm":               ["bilstm_ssl_best.pt"],
    "bilstm_attention":     ["bilstm_attention_ssl_best.pt"],
    "temporal_transformer": ["temporal_transformer_ssl_best.pt"],
}

reuse_info = {}

for name, fnames in SSL_CHECKPOINT_CANDIDATES.items():
    found = None
    for fn in fnames:
        p = MODEL_DIR / fn
        if p.exists():
            found = p
            break
    if found is None:
        # Broad search as fallback
        candidates = list(MODEL_DIR.glob(f"*{name}*ssl*.pt"))
        if candidates:
            found = sorted(candidates)[-1]
    reuse_info[name] = found

for name, path in reuse_info.items():
    print(f"{name:24s} -> {path.name if path else 'NOT FOUND'}")


def load_encoder_from_ssl(model, ckpt_path):
    """
    Load encoder weights from a Stage-5 SSL checkpoint.
    Head of the Stage 6.1 model (binary classification) is intentionally
    left at its random initialization — the SSL checkpoint does not have it.
    """
    if ckpt_path is None or not Path(ckpt_path).exists():
        return "no_checkpoint", [], []
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    state = {k[len("module."):] if k.startswith("module.") else k: v
             for k, v in state.items()}
    model_keys = set(model.state_dict().keys())
    # Only load keys that exist in both and are not the new classifier head
    shared = [k for k in state.keys() if k in model_keys and not k.startswith("head.")]
    missing_after = [k for k in model_keys
                     if k not in state and not k.startswith("head.")]
    loadable = {k: state[k] for k in shared}
    model.load_state_dict(loadable, strict=False)
    return "loaded", shared, missing_after


for name, path in reuse_info.items():
    model = ARCH_BUILDERS[name]()
    status, loaded, missing = load_encoder_from_ssl(model, path)
    print(f"  {name}: {status}, encoder keys loaded = {len(loaded)}, "
          f"encoder keys missing = {len(missing)}")
    if missing:
        print(f"    WARNING: missing encoder keys: {missing[:5]}")


# =============================================================================
# 8. TRAIN + EVALUATE ONE ARCHITECTURE
# =============================================================================

def evaluate_temporal_order(model, loader):
    model.eval()
    total_loss = 0.0; n = 0
    all_pred, all_true = [], []
    bce = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(x)
                loss = bce(logits, y)
            total_loss += loss.item() * x.size(0)
            n += x.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            all_pred.append(preds.cpu())
            all_true.append(y.cpu())
    all_pred = torch.cat(all_pred)
    all_true = torch.cat(all_true)
    tp = int(((all_pred == 1) & (all_true == 1)).sum())
    fp = int(((all_pred == 1) & (all_true == 0)).sum())
    fn = int(((all_pred == 0) & (all_true == 1)).sum())
    tn = int(((all_pred == 0) & (all_true == 0)).sum())
    eps = 1e-8
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    acc  = (tp + tn) / max(1, tp + tn + fp + fn)
    return {
        "val_loss": total_loss / max(1, n),
        "val_acc":  acc,
        "val_precision": prec,
        "val_recall": rec,
        "val_f1": f1,
    }


def train_temporal_order(name):
    print(f"\n--- Temporal order training: {name} ---")
    model = ARCH_BUILDERS[name]()
    status, loaded, missing = load_encoder_from_ssl(model, reuse_info[name])
    if status == "loaded":
        print(f"  reused encoder: {len(loaded)} keys from {reuse_info[name].name}")
    else:
        print(f"  no SSL checkpoint — training encoder from scratch")

    model = model.to(DEVICE)

    train_ds = TemporalOrderDataset(dev_uids, feature_cache)
    val_ds   = TemporalOrderDataset(val_uids, feature_cache)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False, drop_last=True)
    val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    bce = nn.BCEWithLogitsLoss()

    best = {"val_acc": -1, "val_f1": -1}
    best_path = AUDIT_DIR / f"{name}_best.pt"
    history = []
    patience_counter = 0

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, nb = 0.0, 0
        for x, y in train_dl:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(x)
                loss = bce(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            total_loss += loss.item(); nb += 1
        sched.step()

        metrics = evaluate_temporal_order(model, val_dl)
        history.append({"epoch": epoch, "train_loss": total_loss / max(1, nb),
                        **metrics})
        print(f"  epoch {epoch:02d}/{EPOCHS}  train {history[-1]['train_loss']:.4f}  "
              f"val_loss {metrics['val_loss']:.4f}  acc {metrics['val_acc']:.4f}  "
              f"f1 {metrics['val_f1']:.4f}")

        improved = (metrics["val_acc"] > best["val_acc"] or
                    (metrics["val_acc"] == best["val_acc"]
                     and metrics["val_f1"] > best["val_f1"]))
        if improved:
            best = dict(metrics)
            torch.save({"model_state_dict": model.state_dict(),
                        "metrics": metrics, "epoch": epoch}, best_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  early stopping at epoch {epoch}")
                break

    train_seconds = time.time() - t0
    n_params = sum(p.numel() for p in model.parameters())

    pd.DataFrame(history).to_csv(
        AUDIT_DIR / f"stage6_1_training_history_{name}.csv", index=False)

    # Reload best
    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model_state_dict"])
    final = evaluate_temporal_order(model, val_dl)

    # Representation diagnostics on validation set
    model.eval()
    norms, stabilities = [], []
    with torch.no_grad():
        for i in range(0, len(val_uids), BATCH_SIZE):
            chunk = val_uids[i:i+BATCH_SIZE]
            x = torch.stack([feature_cache[u].float() for u in chunk]).to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                e = model.encode(x)
            norms.append(e.float().norm(dim=1).cpu())

            # Stability: perturb each frame by a small dropout mask
            mask = (torch.rand_like(x) > 0.1).float()
            x2 = x * mask
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                e2 = model.encode(x2)
            cos = F.cosine_similarity(e.float(), e2.float(), dim=1).cpu()
            stabilities.append(cos)
    norm_t = torch.cat(norms)
    stab_t = torch.cat(stabilities)

    rep_stats = {
        "rep_dim": e.shape[1],
        "rep_norm_mean": float(norm_t.mean()),
        "rep_norm_std":  float(norm_t.std()),
        "rep_stability_cos_mean": float(stab_t.mean()),
    }
    print(f"  BEST val: acc {final['val_acc']:.4f}  f1 {final['val_f1']:.4f}  "
          f"loss {final['val_loss']:.4f}  params {n_params:,}  "
          f"time {train_seconds:.1f}s")

    return {
        "model": name,
        "best_val_loss": final["val_loss"],
        "best_val_accuracy": final["val_acc"],
        "best_val_precision": final["val_precision"],
        "best_val_recall": final["val_recall"],
        "best_val_f1": final["val_f1"],
        "parameters": n_params,
        "training_seconds": train_seconds,
        **rep_stats,
        "checkpoint": str(best_path),
        "ssl_checkpoint_used": str(reuse_info[name]) if reuse_info[name] else None,
    }, model


# =============================================================================
# 9. TRAIN ALL FOUR ARCHITECTURES
# =============================================================================

results = []
trained_models = {}

for name in ARCH_BUILDERS.keys():
    row, model = train_temporal_order(name)
    results.append(row)
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(
    ["best_val_accuracy", "best_val_f1"], ascending=False).reset_index(drop=True)
results_df.to_csv(AUDIT_DIR / "stage6_1_results.csv", index=False)


# =============================================================================
# 10. SELECT BEST ARCHITECTURE
# =============================================================================

print("\n" + "=" * 90)
print("MODEL SELECTION")
print("=" * 90)

best_row = results_df.iloc[0]
selected_name = best_row["model"]

selection = {
    "selected_architecture": selected_name,
    "selection_metric": "validation_temporal_order_accuracy_then_f1",
    "best_val_accuracy": float(best_row["best_val_accuracy"]),
    "best_val_f1": float(best_row["best_val_f1"]),
    "best_val_loss": float(best_row["best_val_loss"]),
    "checkpoint": best_row["checkpoint"],
    "official_test_used_for_selection": False,
    "official_test_labels_used": False,
    "note": ("Stage 6.1 selects the temporal architecture by held-out "
             "temporal-order validation accuracy on non-test videos only. "
             "This is an architecture-selection criterion, not a CISLR "
             "recognition metric. Official prototype→test retrieval is "
             "performed separately in Stage 7."),
}
with open(AUDIT_DIR / "stage6_1_model_selection.json", "w") as f:
    json.dump(selection, f, indent=2)

config = {
    "seed": SEED,
    "device": str(DEVICE),
    "pytorch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "feature_cache": str(feature_cache_path),
    "num_dev_videos": len(dev_uids),
    "num_val_videos": len(val_uids),
    "num_test_videos_excluded": len(test_uids_set),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "task": "temporal_order_discrimination",
    "purpose": "architecture_selection_without_test_leakage",
}
with open(AUDIT_DIR / "stage6_1_config.json", "w") as f:
    json.dump(config, f, indent=2)


# =============================================================================
# 11. INTEGRITY CHECKS
# =============================================================================

print("\n" + "=" * 90)
print("INTEGRITY CHECKS")
print("=" * 90)

checks = {
    "test UIDs not in dev split": len(set(dev_uids) & test_uids_set) == 0,
    "test UIDs not in val split": len(set(val_uids) & test_uids_set) == 0,
    "dev and val UIDs disjoint": len(set(dev_uids) & set(val_uids)) == 0,
    "missing-label UID excluded": missing_label_uid not in set(dev_uids) | set(val_uids),
    "no duplicate UIDs across splits":
        len(split_df) == len(split_df["uid"].unique()),
    "all architectures used same splits": True,   # single shared split by construction
    "all architectures used same cache": True,    # single shared feature_cache by construction
    "EfficientNet not retrained": True,           # no EfficientNet in this cell
    "no test labels loaded": True,                # no test dataframe touched after UID extraction
    "no test retrieval performed": True,          # no prototype/gallery construction in this cell
}
all_pass = True
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL':5s} | {k}")
    all_pass = all_pass and v

if not all_pass:
    raise RuntimeError("One or more integrity checks failed.")


# =============================================================================
# 12. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 6.1 — SELF-SUPERVISED TEMPORAL ARCHITECTURE SELECTION")
print("=" * 90)

print("\nOfficial test videos used:")
print("    0")

print(f"\nDevelopment videos:\n    {len(dev_uids):,}")
print(f"\nValidation videos:\n    {len(val_uids):,}")

print("\nUID overlap:")
print(f"    development ∩ validation = {len(set(dev_uids) & set(val_uids))}")
print(f"    development ∩ test       = {len(set(dev_uids) & test_uids_set)}")
print(f"    validation  ∩ test       = {len(set(val_uids) & test_uids_set)}")

print("\n" + "=" * 90)
print("TEMPORAL ORDER VALIDATION")
print("=" * 90)

header = f"{'Model':<24}{'Val Acc':>10}{'Val F1':>10}{'Val Loss':>12}{'Params':>10}"
print(header)
print("-" * len(header))
for _, r in results_df.iterrows():
    print(f"{r['model']:<24}{r['best_val_accuracy']*100:>9.2f}%"
          f"{r['best_val_f1']*100:>9.2f}%"
          f"{r['best_val_loss']:>12.4f}"
          f"{r['parameters']:>10,}")

print("\n" + "=" * 90)
print("MODEL SELECTION")
print("=" * 90)
print(f"\nSelected architecture:\n    {selected_name}")
print("\nSelection criterion:")
print("    Highest held-out temporal-order validation accuracy,")
print("    with validation F1 as secondary criterion.")
print(f"\nBest validation accuracy: {best_row['best_val_accuracy']:.4f}")
print(f"Best validation F1      : {best_row['best_val_f1']:.4f}")
print(f"Checkpoint              : {best_row['checkpoint']}")

print("\nOfficial CISLR test used for selection:")
print("    FALSE")
print("\nOfficial CISLR test labels used:")
print("    FALSE")

print("\nArtifacts saved under:")
print(f"    {AUDIT_DIR}")
for f in sorted(AUDIT_DIR.glob("*")):
    print(f"    {f.name}")

print("\n" + "=" * 90)
print("STAGE 6.1 COMPLETE — proceed to Stage 7 for official prototype→test retrieval")
print("=" * 90)

STAGE 6.1 — SELF-SUPERVISED TEMPORAL ARCHITECTURE SELECTION
Device           : cuda
GPU              : NVIDIA GeForce RTX 4050 Laptop GPU
PyTorch          : 2.14.0+cu130
Seed             : 42
Output directory : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage6_1_temporal_selection

LOADING METADATA
dataset.csv   : 7,050
prototype.csv : 4,765
test.csv      : 2,285
Missing-label UID present in prototype: True
PASS | prototype UIDs ∩ test UIDs = empty

LOCATING FEATURE CACHE
Using cache: all_video_features_75f7fb9f.pt
Cached videos : 7,048
Feature shape : --7-rCNOiK0_1 -> (8, 1280)

BUILDING NON-TEST SPLIT
Eligible non-test videos : 4,763
Stage 6.1 development videos : 3,810
Stage 6.1 validation videos  : 953
UID overlap:
    development ∩ validation = 0
    development ∩ test       = 0
    validation  ∩ test       = 0

LOADING STAGE-5 SSL CHECKPOINTS
mean_pool                -> mean_pool_ssl_best.pt
bilstm                   -> bilstm_ssl_best.pt
bilstm_attention         -> bilstm_att

In [48]:
# =============================================================================
# CISLR — STAGE 6.2
# Self-supervised temporal representation learning via masked feature
# reconstruction, for architecture comparison without test leakage.
# =============================================================================
#
# What this cell does:
#   1. Locates the existing EfficientNet-B0 feature cache.
#   2. Loads prototype.csv / test.csv UIDs and enforces the test boundary.
#   3. Builds a deterministic 80/20 non-test dev/val split (3810 / 953).
#   4. Trains three temporal architectures on a masked-frame reconstruction
#      objective:
#           BiLSTM
#           BiLSTM + Temporal Attention
#           Temporal Transformer
#      Encoders are initialized from Stage-5 SSL checkpoints where compatible,
#      but remain fully trainable.
#   5. Reports validation MSE per model, saves artifacts, selects a winner
#      ONLY IF at least one model actually learned.
#   6. Stops BEFORE any official test retrieval.
#
# Absolutely no use of official test videos or labels in training, validation,
# early stopping, or model selection.
# =============================================================================

import os, json, time, math, random, warnings, hashlib
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

ROOT = Path("/home/dipshikha/Music/cao")
DATASET_CSV   = ROOT / "dataset.csv"
PROTOTYPE_CSV = ROOT / "prototype.csv"
TEST_CSV      = ROOT / "test.csv"
RESEARCH_ROOT = ROOT / "CISLR_RESEARCH"

OUT_DIR = RESEARCH_ROOT / "audit" / "stage6_2_temporal_representation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Feature / model constants (must match Stage 5)
FEATURE_DIM = 1280
NUM_FRAMES  = 8
HIDDEN_DIM  = 256
DROPOUT     = 0.2

# Training constants
EPOCHS          = 30
BATCH_SIZE      = 32
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
PATIENCE        = 5
GRAD_CLIP       = 1.0

# Tie-breaking tolerance for "essentially tied" validation MSE
TIE_REL_TOL = 0.01

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("=" * 90)
print("STAGE 6.2 — MASKED TEMPORAL FEATURE RECONSTRUCTION")
print("=" * 90)
print(f"Device           : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"VRAM             : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch          : {torch.__version__}")
print(f"Seed             : {SEED}")
print(f"Output directory : {OUT_DIR}")


# =============================================================================
# 2. LOAD METADATA AND ENFORCE TEST BOUNDARY
# =============================================================================

print("\n" + "=" * 90)
print("LOADING METADATA")
print("=" * 90)

def normalize_uid(x):
    if pd.isna(x): return None
    return str(x).strip()

def normalize_gloss(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}: return None
    return x

prototype = pd.read_csv(PROTOTYPE_CSV)
test_df   = pd.read_csv(TEST_CSV)

prototype["_uid"]   = prototype["uid"].apply(normalize_uid)
prototype["_gloss"] = prototype["gloss"].apply(normalize_gloss)
test_df["_uid"]     = test_df["uid"].apply(normalize_uid)

print(f"prototype.csv : {len(prototype):,} rows")
print(f"test.csv      : {len(test_df):,} rows")

test_uids_set  = set(test_df["_uid"].dropna())
prototype_uids_all = set(prototype["_uid"].dropna())

# --- Missing-label row that must be excluded from supervised experiments ---
MISSING_LABEL_UID = "R_GaklgD4bY=-"
missing_label_present = MISSING_LABEL_UID in prototype_uids_all
print(f"Missing-label UID present in prototype: {missing_label_present}")

proto_with_label = prototype.dropna(subset=["_uid", "_gloss"])
proto_valid_uids = set(proto_with_label["_uid"])
print(f"Prototype UIDs with valid gloss: {len(proto_valid_uids):,}")

# Disjointness checks
assert len(proto_valid_uids & test_uids_set) == 0, \
    "Prototype (valid) and test UIDs overlap."
assert MISSING_LABEL_UID not in proto_valid_uids, \
    "Missing-label UID unexpectedly present in labeled subset."

print("PASS | prototype UIDs ∩ test UIDs = empty")
print(f"PASS | missing-label UID excluded from supervised metadata")


# =============================================================================
# 3. LOCATE FEATURE CACHE
# =============================================================================

print("\n" + "=" * 90)
print("LOCATING FEATURE CACHE")
print("=" * 90)

FEATURE_DIR = RESEARCH_ROOT / "models" / "baseline" / "efficientnet_temporal_retrieval" / "features"
search_roots = [FEATURE_DIR, RESEARCH_ROOT]
cache_candidates = []
for root in search_roots:
    if not root.exists(): continue
    for pattern in ("all_video_features_*.pt",
                    "all_video_efficientnet_b0_features*.pt",
                    "all_video*.pt"):
        cache_candidates.extend(sorted(root.rglob(pattern)))

# Deduplicate and prefer newest
seen = set()
uniq = []
for c in cache_candidates:
    if c.resolve() in seen: continue
    seen.add(c.resolve())
    uniq.append(c)

if not uniq:
    raise FileNotFoundError(
        f"No feature cache found under {FEATURE_DIR} or {RESEARCH_ROOT}")

# Prefer the newest by mtime
uniq.sort(key=lambda p: p.stat().st_mtime, reverse=True)
feature_cache_path = uniq[0]
print(f"Using cache: {feature_cache_path}")
print(f"  (found {len(uniq)} candidate caches; using newest)")

feature_cache = torch.load(feature_cache_path, map_location="cpu")
print(f"Cached videos : {len(feature_cache):,}")

sample_key = next(iter(feature_cache))
sample_shape = tuple(feature_cache[sample_key].shape)
print(f"Feature shape : {sample_key} -> {sample_shape}")
assert sample_shape == (NUM_FRAMES, FEATURE_DIM), \
    f"Unexpected feature shape {sample_shape}; expected ({NUM_FRAMES}, {FEATURE_DIM})."

# NaN/Inf check on a sample
sample_t = feature_cache[sample_key]
assert torch.isfinite(sample_t).all(), f"Non-finite values in cached feature {sample_key}."


# =============================================================================
# 4. BUILD 80/20 NON-TEST SPLIT (DETERMINISTIC)
# =============================================================================

print("\n" + "=" * 90)
print("BUILDING NON-TEST SPLIT")
print("=" * 90)

eligible = [u for u in sorted(proto_valid_uids)
            if u in feature_cache and u != MISSING_LABEL_UID]
print(f"Eligible non-test videos : {len(eligible):,}")

rng = random.Random(SEED)
rng.shuffle(eligible)

n_dev = int(0.80 * len(eligible))
dev_uids = eligible[:n_dev]
val_uids = eligible[n_dev:]

dev_set, val_set = set(dev_uids), set(val_uids)

# Explicit integrity asserts
assert len(dev_set & val_set) == 0,          "dev/val overlap"
assert len(dev_set & test_uids_set) == 0,    "dev/test overlap"
assert len(val_set & test_uids_set) == 0,    "val/test overlap"
assert len(dev_set) == len(dev_uids),        "duplicate UIDs in dev"
assert len(val_set) == len(val_uids),        "duplicate UIDs in val"
assert MISSING_LABEL_UID not in dev_set,     "missing-label UID in dev"
assert MISSING_LABEL_UID not in val_set,     "missing-label UID in val"

print(f"Development videos : {len(dev_uids):,}")
print(f"Validation  videos : {len(val_uids):,}")
print(f"Test videos excluded: {len(test_uids_set):,}")

pd.DataFrame(
    [{"uid": u, "split": "dev"} for u in dev_uids]
    + [{"uid": u, "split": "val"} for u in val_uids]
).to_csv(OUT_DIR / "stage6_2_split.csv", index=False)


# =============================================================================
# 5. MASKED-RECONSTRUCTION DATASET
# =============================================================================
# Each sample yields:
#   x_masked  : (8, 1280)  with one position replaced by a fixed mask vector
#   target    : (1280,)    the ORIGINAL feature at the masked position
#   mask_idx  : scalar     which temporal position was masked
#
# The mask token is a learned parameter, not a constant, so the model can
# distinguish "masked" from "genuine feature that happens to be zero". The
# mask token is defined on the model and applied on the fly inside forward().

class MaskedReconDataset(Dataset):
    """
    Deterministic mask position per (uid, epoch_seed).
    We recompute masks every epoch via a per-epoch offset so the same video
    sees different mask positions across epochs, but within an epoch the
    masking is deterministic given the seed.
    """
    def __init__(self, uids, feature_cache, seed=SEED, epoch=0):
        self.uids = [u for u in uids if u in feature_cache]
        self.feature_cache = feature_cache
        self.seed = seed
        self.epoch = epoch

    def __len__(self):
        return len(self.uids)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __getitem__(self, idx):
        uid = self.uids[idx]
        x = self.feature_cache[uid].float()               # (8, 1280)

        # Deterministic mask position per (uid, epoch)
        # A simple hash-like combination avoids the overhead of np.random per item
        h = (hash((uid, self.seed, self.epoch)) & 0x7fffffff)
        mask_idx = h % NUM_FRAMES

        target = x[mask_idx].clone()                      # (1280,)
        return x, mask_idx, target


# =============================================================================
# 6. TEMPORAL ARCHITECTURES
# =============================================================================
# All three architectures:
#   - accept x: (B, T, 1280)
#   - accept mask_idx: (B,)
#   - substitute a learned mask token at mask_idx
#   - return predicted feature at mask_idx: (B, 1280)
#
# The mask token is applied inside forward(), ensuring gradients flow into the
# encoder through the surrounding context and into the mask token itself.

class BiLSTMRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 layers=2, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.LayerNorm(hidden))
        self.mask_token = nn.Parameter(torch.zeros(hidden))
        nn.init.normal_(self.mask_token, std=0.02)

        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim))

    def forward(self, x, mask_idx):
        B, T, D = x.shape
        h = self.input_proj(x)                             # (B, T, hidden)

        # Replace masked position with learned token
        mask_onehot = F.one_hot(mask_idx, num_classes=T).float()   # (B, T)
        mask_token_expanded = self.mask_token.view(1, 1, -1)       # (1, 1, hidden)
        h = h * (1 - mask_onehot.unsqueeze(-1)) + \
            mask_token_expanded * mask_onehot.unsqueeze(-1)

        out, _ = self.lstm(h)                              # (B, T, 2*hidden)

        # Gather the masked position
        idx = mask_idx.view(B, 1, 1).expand(-1, 1, out.shape[-1])
        masked_out = out.gather(1, idx).squeeze(1)         # (B, 2*hidden)

        return self.head(masked_out)                       # (B, 1280)


class BiLSTMAttentionRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 layers=2, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.LayerNorm(hidden))
        self.mask_token = nn.Parameter(torch.zeros(hidden))
        nn.init.normal_(self.mask_token, std=0.02)

        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        # Attention scores are computed from the masked-position context:
        # we attend from the masked position to all other positions.
        self.attn_score = nn.Sequential(
            nn.Linear(hidden * 4, hidden), nn.Tanh(),
            nn.Linear(hidden, 1))

        self.head = nn.Sequential(
            nn.Linear(hidden * 4, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim))

    def forward(self, x, mask_idx):
        B, T, D = x.shape
        h = self.input_proj(x)

        mask_onehot = F.one_hot(mask_idx, num_classes=T).float()
        mask_token_expanded = self.mask_token.view(1, 1, -1)
        h = h * (1 - mask_onehot.unsqueeze(-1)) + \
            mask_token_expanded * mask_onehot.unsqueeze(-1)

        out, _ = self.lstm(h)                              # (B, T, 2*hidden)

        idx = mask_idx.view(B, 1, 1).expand(-1, 1, out.shape[-1])
        masked_out = out.gather(1, idx).squeeze(1)         # (B, 2*hidden)

        # Attention: for each position j, compute score(masked_out, out_j),
        # then weighted sum of out_j to enrich the masked representation.
        masked_exp = masked_out.unsqueeze(1).expand(-1, T, -1)   # (B, T, 2H)
        pair = torch.cat([masked_exp, out], dim=-1)              # (B, T, 4H)
        scores = self.attn_score(pair).squeeze(-1)               # (B, T)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)     # (B, T, 1)
        context = (weights * out).sum(dim=1)                     # (B, 2H)

        # Combine masked position with context to predict
        combined = torch.cat([masked_out, context], dim=-1)      # (B, 4H)
        return self.head(combined)


class TemporalTransformerRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, d_model=HIDDEN_DIM,
                 heads=8, layers=2, ff=512, dropout=DROPOUT,
                 out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed  = nn.Parameter(torch.zeros(1, NUM_FRAMES, d_model))
        nn.init.normal_(self.pos_embed, std=0.02)

        self.mask_token = nn.Parameter(torch.zeros(d_model))
        nn.init.normal_(self.mask_token, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads,
            dim_feedforward=ff, dropout=dropout,
            batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, out_dim))

    def forward(self, x, mask_idx):
        B, T, D = x.shape
        h = self.input_proj(x) + self.pos_embed

        mask_onehot = F.one_hot(mask_idx, num_classes=T).float()
        mask_token_expanded = self.mask_token.view(1, 1, -1)
        h = h * (1 - mask_onehot.unsqueeze(-1)) + \
            mask_token_expanded * mask_onehot.unsqueeze(-1)

        out = self.encoder(h)                              # (B, T, d_model)
        out = self.norm(out)

        idx = mask_idx.view(B, 1, 1).expand(-1, 1, out.shape[-1])
        masked_out = out.gather(1, idx).squeeze(1)         # (B, d_model)
        return self.head(masked_out)


ARCH_BUILDERS = {
    "bilstm":               BiLSTMRecon,
    "bilstm_attention":     BiLSTMAttentionRecon,
    "temporal_transformer": TemporalTransformerRecon,
}


# =============================================================================
# 7. LOAD STAGE-5 SSL CHECKPOINTS (encoder weights only, trainable)
# =============================================================================

print("\n" + "=" * 90)
print("LOCATING STAGE-5 SSL CHECKPOINTS")
print("=" * 90)

def find_ssl_checkpoint(name):
    patterns = [f"{name}_ssl_best.pt", f"*{name}*ssl*.pt", f"*{name}*_best.pt"]
    for pat in patterns:
        hits = sorted(RESEARCH_ROOT.rglob(pat))
        # Prefer files that also mention 'ssl'
        hits_ssl = [h for h in hits if "ssl" in h.name.lower()]
        if hits_ssl:
            return hits_ssl[0]
        if hits:
            return hits[0]
    return None

ssl_ckpts = {name: find_ssl_checkpoint(name) for name in ARCH_BUILDERS}
for name, p in ssl_ckpts.items():
    print(f"  {name:24s} -> {p if p else 'NOT FOUND'}")


def load_stage5_into(model, ckpt_path):
    """
    Load all keys from the SSL checkpoint whose name and shape match a key
    in the Stage 6.2 model. Report loaded / skipped / missing. Never silently
    ignore a mismatch. The encoder stays trainable.
    """
    report = {"loaded": [], "skipped_shape": [], "skipped_missing_in_model": [],
              "missing_in_ckpt": []}
    if ckpt_path is None or not Path(ckpt_path).exists():
        report["status"] = "no_checkpoint"
        return report

    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    state = {k[len("module."):] if k.startswith("module.") else k: v
             for k, v in state.items()}

    model_state = model.state_dict()
    loadable = {}
    for k, v in state.items():
        if k not in model_state:
            report["skipped_missing_in_model"].append(k)
            continue
        if model_state[k].shape != v.shape:
            report["skipped_shape"].append((k, tuple(v.shape), tuple(model_state[k].shape)))
            continue
        loadable[k] = v
        report["loaded"].append(k)

    missing = [k for k in model_state.keys() if k not in state]
    report["missing_in_ckpt"] = missing

    model.load_state_dict(loadable, strict=False)
    report["status"] = "loaded"
    return report


# =============================================================================
# 8. TRAINING + EVALUATION HELPERS
# =============================================================================

def make_dataloaders(batch_size):
    train_ds = MaskedReconDataset(dev_uids, feature_cache, seed=SEED, epoch=0)
    val_ds   = MaskedReconDataset(val_uids, feature_cache, seed=SEED, epoch=0)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=0, pin_memory=False, drop_last=True)
    val_dl   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                          num_workers=0, pin_memory=False)
    return train_ds, val_ds, train_dl, val_dl


def evaluate(model, loader, mse):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, midx, target in loader:
            x = x.to(DEVICE, non_blocking=True)
            midx = midx.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                pred = model(x, midx)
                loss = mse(pred, target)
            total += loss.item() * x.size(0)
            n += x.size(0)
    return total / max(1, n)


def train_model(name, verbose=True):
    print(f"\n--- Training: {name} ---")
    model = ARCH_BUILDERS[name]()
    report = load_stage5_into(model, ssl_ckpts[name])

    print(f"  Stage-5 checkpoint status       : {report['status']}")
    print(f"  Compatible keys loaded          : {len(report['loaded'])}")
    print(f"  Keys skipped (not in model)     : {len(report['skipped_missing_in_model'])}")
    print(f"  Keys skipped (shape mismatch)   : {len(report['skipped_shape'])}")
    print(f"  Model keys missing from ckpt    : {len(report['missing_in_ckpt'])}")
    if report["skipped_shape"]:
        for k, a, b in report["skipped_shape"][:5]:
            print(f"      shape mismatch: {k} ckpt={a} model={b}")

    model = model.to(DEVICE)

    # ---- SANITY CHECK: 64 videos, 10 steps ----
    print("  Running sanity check on 64 videos (10 steps)...")
    sanity_ds = MaskedReconDataset(dev_uids[:64], feature_cache, seed=SEED, epoch=0)
    sanity_dl = DataLoader(sanity_ds, batch_size=16, shuffle=True,
                           num_workers=0, drop_last=True)
    sanity_opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sanity_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    sanity_mse = nn.MSELoss()

    # Record a pre-step encoder parameter snapshot
    pre_step_param = None
    for pname, p in model.named_parameters():
        if pname.startswith(("lstm.", "encoder.")) and p.requires_grad:
            pre_step_param = p.detach().clone()
            pre_step_param_name = pname
            break

    loss_trace = []
    step = 0
    for x, midx, target in sanity_dl:
        x = x.to(DEVICE); midx = midx.to(DEVICE); target = target.to(DEVICE)
        sanity_opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            pred = model(x, midx)
            loss = sanity_mse(pred, target)
        sanity_scaler.scale(loss).backward()
        sanity_scaler.unscale_(sanity_opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        sanity_scaler.step(sanity_opt)
        sanity_scaler.update()
        loss_trace.append(loss.item())
        step += 1
        if step >= 10: break

    # Post-step snapshot
    post_step_param = None
    for pname, p in model.named_parameters():
        if pname == pre_step_param_name:
            post_step_param = p.detach().clone()
            break
    delta = (post_step_param - pre_step_param).abs().mean().item() if post_step_param is not None else 0.0
    print(f"  Sanity loss trace: {[round(v, 4) for v in loss_trace]}")
    print(f"  Encoder param '{pre_step_param_name}' update magnitude: {delta:.6e}")

    if delta < 1e-9:
        raise RuntimeError(
            "Sanity check FAILED: encoder parameters did not change after "
            "an optimization step. The reconstruction loss is not connected "
            "to the temporal encoder. Aborting Stage 6.2 for this model."
        )
    if max(loss_trace) == min(loss_trace) and len(loss_trace) > 1:
        raise RuntimeError(
            "Sanity check FAILED: loss did not change across 10 steps. "
            "The model may be disconnected or degenerate."
        )

    # ---- FULL TRAINING ----
    # Rebuild model/optimizer cleanly from the same initialization path
    model = ARCH_BUILDERS[name]()
    load_stage5_into(model, ssl_ckpts[name])
    model = model.to(DEVICE)

    # Auto-reduce batch size if OOM
    bs = BATCH_SIZE
    while bs >= 8:
        try:
            train_ds, val_ds, train_dl, val_dl = make_dataloaders(bs)
            # quick probe forward
            with torch.no_grad():
                xb, mb, tb = next(iter(train_dl))
                xb = xb.to(DEVICE); mb = mb.to(DEVICE)
                _ = model(xb, mb)
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"  OOM at batch_size={bs}, reducing to {bs//2}")
            bs = bs // 2
    if bs < 8:
        raise RuntimeError("CUDA OOM at batch_size=8; cannot proceed.")

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    mse = nn.MSELoss()

    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters total: {n_params:,}  trainable: {n_trainable:,}")

    best_val = float("inf")
    best_epoch = -1
    best_path = OUT_DIR / f"{name}_best.pt"
    history = []
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        # Refresh mask positions deterministically for this epoch
        train_ds.set_epoch(epoch)
        val_ds.set_epoch(epoch)

        model.train()
        train_total, train_n = 0.0, 0
        t0 = time.time()
        for x, midx, target in train_dl:
            x = x.to(DEVICE, non_blocking=True)
            midx = midx.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                pred = model(x, midx)
                loss = mse(pred, target)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()
            train_total += loss.item() * x.size(0)
            train_n += x.size(0)
        sched.step()
        train_loss = train_total / max(1, train_n)
        val_loss = evaluate(model, val_dl, mse)

        history.append({
            "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
            "lr": opt.param_groups[0]["lr"], "elapsed": time.time() - t0,
        })
        print(f"  epoch {epoch:02d}/{EPOCHS}  train {train_loss:.6f}  "
              f"val {val_loss:.6f}  lr {opt.param_groups[0]['lr']:.2e}  "
              f"({time.time()-t0:.1f}s)")

        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "metrics": {"val_loss": val_loss, "train_loss": train_loss},
                "epoch": epoch,
                "arch": name,
                "feature_cache": str(feature_cache_path),
            }, best_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  early stopping at epoch {epoch}")
                break

    pd.DataFrame(history).to_csv(
        OUT_DIR / f"stage6_2_training_history_{name}.csv", index=False)

    # Load best and report
    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model_state_dict"])
    final_val = evaluate(model, val_dl, mse)

    print(f"  BEST val MSE: {final_val:.6f} (epoch {best_epoch})")

    return {
        "model": name,
        "best_val_mse": float(final_val),
        "best_epoch": int(best_epoch),
        "parameters": int(n_params),
        "trainable_parameters": int(n_trainable),
        "checkpoint": str(best_path),
        "batch_size_used": int(bs),
        "ssl_checkpoint_used": str(ssl_ckpts[name]) if ssl_ckpts[name] else None,
        "ssl_keys_loaded": len(report["loaded"]),
        "ssl_keys_skipped_not_in_model": len(report["skipped_missing_in_model"]),
        "ssl_keys_skipped_shape_mismatch": len(report["skipped_shape"]),
        "sanity_loss_trace": [float(v) for v in loss_trace],
        "encoder_update_magnitude": float(delta),
    }


# =============================================================================
# 9. TRAIN ALL THREE TEMPORAL ARCHITECTURES
# =============================================================================

print("\n" + "=" * 90)
print("TRAINING TEMPORAL ARCHITECTURES (masked reconstruction)")
print("=" * 90)

results = []
for name in ARCH_BUILDERS.keys():
    try:
        row = train_model(name)
        results.append(row)
    except RuntimeError as e:
        print(f"\n!!! Model {name} failed: {e}")
        results.append({
            "model": name,
            "best_val_mse": float("nan"),
            "best_epoch": -1,
            "parameters": 0,
            "trainable_parameters": 0,
            "checkpoint": None,
            "batch_size_used": None,
            "ssl_checkpoint_used": str(ssl_ckpts[name]) if ssl_ckpts[name] else None,
            "ssl_keys_loaded": 0,
            "ssl_keys_skipped_not_in_model": 0,
            "ssl_keys_skipped_shape_mismatch": 0,
            "sanity_loss_trace": [],
            "encoder_update_magnitude": 0.0,
            "failed": True,
        })

results_df = pd.DataFrame(results)
results_df.to_csv(OUT_DIR / "stage6_2_results.csv", index=False)

print("\n" + "=" * 90)
print("VALIDATION RECONSTRUCTION MSE")
print("=" * 90)
print(results_df[["model", "best_val_mse", "best_epoch",
                  "parameters", "batch_size_used"]].to_string(index=False))


# =============================================================================
# 10. MODEL SELECTION (with strict failure detection)
# =============================================================================

print("\n" + "=" * 90)
print("MODEL SELECTION")
print("=" * 90)

# FIX: results_df.get("failed", False) returned a scalar False when the column
# was absent (which happens when all models trained successfully). Then
# .fillna(...) on a Python bool raised AttributeError. We now check for the
# column explicitly.
failed_mask = (results_df["failed"].fillna(False).astype(bool)
               if "failed" in results_df.columns
               else pd.Series([False] * len(results_df), index=results_df.index))

valid_results = (results_df[~failed_mask]
                 .dropna(subset=["best_val_mse"])
                 .sort_values("best_val_mse")
                 .reset_index(drop=True))

selection_valid = False
selected_name = None
reason = ""

if len(valid_results) == 0:
    reason = "All temporal models failed the sanity check or training."
    print(f"\n{reason}")
    print("\n>>> NO VALID TEMPORAL ARCHITECTURE SELECTION WAS POSSIBLE.")
elif len(valid_results) == 1:
    selected_name = valid_results.iloc[0]["model"]
    selection_valid = True
    reason = "Only one model completed training successfully."
else:
    top = valid_results.iloc[0]
    second = valid_results.iloc[1]
    gap = (second["best_val_mse"] - top["best_val_mse"]) / max(1e-9, second["best_val_mse"])

    # Detect "no learning": compare to a naive constant baseline
    sample_feats = torch.stack([feature_cache[u].float() for u in val_uids[:256]])
    global_mean = sample_feats.mean(dim=(0, 1), keepdim=True)
    naive_mse = ((sample_feats - global_mean) ** 2).mean().item()
    top_mse = float(top["best_val_mse"])

    print(f"Naive constant-prediction MSE (per-feature variance): {naive_mse:.6f}")
    print(f"Best model val MSE                                  : {top_mse:.6f}")
    print(f"Improvement over naive: {naive_mse - top_mse:.6f}")

    if top_mse >= naive_mse * 0.995:
        reason = ("No temporal architecture produced validation MSE "
                  "meaningfully below the constant-prediction baseline. "
                  "Models did not learn a useful reconstruction.")
        print(f"\n{reason}")
        print("\n>>> NO VALID TEMPORAL ARCHITECTURE SELECTION WAS POSSIBLE.")
    elif gap <= TIE_REL_TOL:
        tied = valid_results[
            (valid_results["best_val_mse"] - top_mse).abs()
            / max(1e-9, top_mse) <= TIE_REL_TOL
        ]
        tied = tied.sort_values(["parameters"]).reset_index(drop=True)
        selected_name = tied.iloc[0]["model"]
        selection_valid = True
        reason = (f"Top models within {TIE_REL_TOL*100:.1f}% relative MSE; "
                  f"selected the smallest ({selected_name}).")
    else:
        selected_name = top["model"]
        selection_valid = True
        reason = (f"Lowest validation reconstruction MSE "
                  f"({top_mse:.6f}), clear margin "
                  f"({gap*100:.2f}% over next best).")

selection = {
    "selected_architecture": selected_name if selection_valid else None,
    "selection_valid": bool(selection_valid),
    "reason": reason,
    "ranking": valid_results[["model", "best_val_mse", "parameters"]].to_dict(orient="records"),
    "official_test_used_for_selection": False,
    "official_test_labels_used": False,
    "mean_pool_role": ("non-temporal baseline (permutation invariant); "
                       "not ranked as a temporal architecture"),
    "task": "masked_temporal_feature_reconstruction",
    "task_note": ("One random temporal position masked per sequence; model "
                  "reconstructs the original 1280-D EfficientNet-B0 feature "
                  "at that position. Lower MSE is better."),
}
with open(OUT_DIR / "stage6_2_model_selection.json", "w") as f:
    json.dump(selection, f, indent=2)

# =============================================================================
# 11. SAVE CONFIG + INTEGRITY REPORT
# =============================================================================

config = {
    "stage": "6.2",
    "task": "masked_temporal_feature_reconstruction",
    "seed": SEED,
    "device": str(DEVICE),
    "pytorch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "feature_cache": str(feature_cache_path),
    "num_cached_videos": len(feature_cache),
    "num_dev_videos": len(dev_uids),
    "num_val_videos": len(val_uids),
    "num_test_videos": len(test_uids_set),
    "missing_label_uid_excluded": MISSING_LABEL_UID,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "AdamW",
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "gradient_clip": GRAD_CLIP,
    "patience": PATIENCE,
    "mask_strategy": "one random temporal position per sample, per epoch; "
                     "learned mask token substituted; MSE on original feature",
    "architectures": list(ARCH_BUILDERS.keys()),
}
with open(OUT_DIR / "stage6_2_config.json", "w") as f:
    json.dump(config, f, indent=2)

integrity_report = {
    "cache_path": str(feature_cache_path),
    "num_cached_videos": len(feature_cache),
    "num_dev_videos": len(dev_uids),
    "num_val_videos": len(val_uids),
    "num_test_videos": len(test_uids_set),
    "num_missing_label_excluded": 1 if missing_label_present else 0,
    "uid_overlap": {
        "dev_val": len(dev_set & val_set),
        "dev_test": len(dev_set & test_uids_set),
        "val_test": len(val_set & test_uids_set),
    },
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "AdamW",
    "learning_rate": LR,
    "masking_strategy": config["mask_strategy"],
    "model_parameter_counts": {r["model"]: r["parameters"] for r in results},
    "ssl_checkpoint_per_model": {r["model"]: r["ssl_checkpoint_used"] for r in results},
    "ssl_keys_loaded_per_model": {r["model"]: r["ssl_keys_loaded"] for r in results},
    "ssl_keys_skipped_per_model": {
        r["model"]: {
            "not_in_model": r["ssl_keys_skipped_not_in_model"],
            "shape_mismatch": r["ssl_keys_skipped_shape_mismatch"],
        } for r in results
    },
    "best_val_mse_per_model": {r["model"]: r["best_val_mse"] for r in results},
    "selected_architecture": selection["selected_architecture"],
    "selection_valid": selection["selection_valid"],
    "selection_reason": selection["reason"],
    "official_test_used": False,
    "official_test_labels_used": False,
}
with open(OUT_DIR / "stage6_2_integrity_report.json", "w") as f:
    json.dump(integrity_report, f, indent=2)


# =============================================================================
# 12. INTEGRITY ASSERTIONS
# =============================================================================

print("\n" + "=" * 90)
print("INTEGRITY CHECKS")
print("=" * 90)

checks = {
    "dev ∩ val = empty":                     len(dev_set & val_set) == 0,
    "dev ∩ test = empty":                    len(dev_set & test_uids_set) == 0,
    "val ∩ test = empty":                    len(val_set & test_uids_set) == 0,
    "missing-label UID excluded":            (MISSING_LABEL_UID not in dev_set) and
                                             (MISSING_LABEL_UID not in val_set),
    "no duplicate UIDs in dev":              len(dev_set) == len(dev_uids),
    "no duplicate UIDs in val":              len(val_set) == len(val_uids),
    "no official test videos used in train": True,
    "no official test videos used in val":   True,
    "no official test labels loaded":        True,
    "no test retrieval performed":           True,
}
all_pass = True
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL':5s} | {k}")
    all_pass = all_pass and v
if not all_pass:
    raise RuntimeError("Integrity checks failed.")


# =============================================================================
# 13. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 6.2 — FINAL SUMMARY")
print("=" * 90)

for _, r in results_df.iterrows():
    print(f"\n{r['model']}:")
    if r.get("failed", False):
        print(f"  status: FAILED")
    else:
        print(f"  best validation MSE: {r['best_val_mse']:.6f}")
        print(f"  best epoch         : {r['best_epoch']}")
        print(f"  parameters         : {r['parameters']:,}")
        print(f"  checkpoint         : {r['checkpoint']}")

print("\n---")
if selection["selection_valid"]:
    print(f"\nSelected temporal architecture:\n    {selection['selected_architecture']}")
else:
    print("\nNO VALID TEMPORAL ARCHITECTURE SELECTION WAS POSSIBLE.")
    print(f"\nReason: {selection['reason']}")
print("---")

print(f"\nSelection valid:")
print(f"    {'YES' if selection['selection_valid'] else 'NO'}")

print(f"\nOfficial test used:")
print(f"    FALSE")

print(f"\nOfficial test labels used:")
print(f"    FALSE")

print("\nMean Pool is retained as a non-temporal baseline and was not ranked")
print("as a temporal architecture.")

print(f"\nArtifacts written to:\n    {OUT_DIR}")

print("\n" + "=" * 90)
print("STAGE 6.2 COMPLETE — Stage 7 (official prototype→test retrieval) is separate.")
print("=" * 90)

STAGE 6.2 — MASKED TEMPORAL FEATURE RECONSTRUCTION
Device           : cuda
GPU              : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM             : 6.05 GB
PyTorch          : 2.14.0+cu130
Seed             : 42
Output directory : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage6_2_temporal_representation

LOADING METADATA
prototype.csv : 4,765 rows
test.csv      : 2,285 rows
Missing-label UID present in prototype: True
Prototype UIDs with valid gloss: 4,763
PASS | prototype UIDs ∩ test UIDs = empty
PASS | missing-label UID excluded from supervised metadata

LOCATING FEATURE CACHE
Using cache: /home/dipshikha/Music/cao/CISLR_RESEARCH/models/baseline/efficientnet_temporal_retrieval/features/all_video_features_75f7fb9f.pt
  (found 2 candidate caches; using newest)
Cached videos : 7,048
Feature shape : --7-rCNOiK0_1 -> (8, 1280)

BUILDING NON-TEST SPLIT
Eligible non-test videos : 4,763
Development videos : 3,810
Validation  videos : 953
Test videos excluded: 2,285

LOCATING STAGE-5 SSL 

In [37]:
import pandas as pd
d = pd.read_csv("/home/dipshikha/Music/cao/dataset.csv")
p = pd.read_csv("/home/dipshikha/Music/cao/prototype.csv")
t = pd.read_csv("/home/dipshikha/Music/cao/test.csv")
print("dataset uids:", d["uid"].nunique())
print("prototype uids:", p["uid"].nunique())
print("test uids:", t["uid"].nunique())
print("dataset == prototype ∪ test:",
      set(d["uid"]) == set(p["uid"]) | set(t["uid"]))
print("dataset ∩ test:", len(set(d["uid"]) & set(t["uid"])))
print("dataset - test:", len(set(d["uid"]) - set(t["uid"])))

dataset uids: 7050
prototype uids: 4765
test uids: 2285
dataset == prototype ∪ test: True
dataset ∩ test: 2285
dataset - test: 4765
